<!-- notebook-header -->
<a id="top"></a>

# 9. Graphs

*LeetCode Playbook · notebook 9 of 12* · BFS and DFS, ordering and connectivity, weighted shortest paths and spanning trees.

**In this notebook:** [Graphs I: BFS & DFS](#s17) · [Graphs II: Ordering & Connectivity](#s18) · [Graphs III: Weighted Paths & MST](#s19)

← [Backtracking (and a short DP map)](08_Backtracking.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Matrices, Math & Bits](10_Matrices_Math_Bits.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s17"></a>

## Graphs I: BFS & DFS

> A graph is just *things* (nodes) and *you can get from this one to that one* (edges). BFS spreads from the start like a ripple in a pond, one ring of equal distance at a time; DFS runs down one corridor to its end before backing up. Both visit every node they can reach exactly once, because they mark a node the moment they discover it.

**Reach for it when** the problem talks about cells connected up, down, left and right, islands or regions, or something spreading in rounds: rot, fire, a signal. Reach for it too when it asks for the fewest steps, moves, jumps or transformations and every move costs the same, when it asks "can I reach", when a structure with cycles must be copied, or when a situation changes one move at a time: a board, a word, a lock.

**In this repo:** `graphs/` (19 of its 45 problems; Graphs II and III take the rest) · bank: `practice/simple/41_number_of_islands.py`, `practice/simple/42_clone_graph.py`, `practice/simple/44_rotting_oranges.py` · basics: `practice/simple/basics/graphs/01_adjacency_list_bfs_dfs.py`, `practice/simple/basics/searches/03_bfs_grid_shortest_path.py`, `practice/simple/basics/searches/04_dfs_recursive_and_iterative.py`, `practice/simple/basics/searches/05_connected_components.py`, `practice/simple/basics/searches/06_multi_source_bfs_01_matrix.py`

### The picture

Half of every graph problem is one sentence: *what is a node, and what is an edge?* Once you can say it, the code is one of two loops.

In a grid of land and water, a node is a cell `(r, c)` and an edge is a step up, down, left or right onto land, so "how many groups?" is a flood fill, by DFS or by BFS. When rot spreads every minute, an edge is one minute of spreading, and "when is the last cell reached?" is a BFS from every rotten cell at once.

The node is whatever changes from one move to the next. When a word changes one letter at a time, the node is a word, joined to every word one letter away. When tiles slide into a gap, it is the whole board as a string; when keys open locks, a cell with the keys in hand; when the question counts buses, a bus route, joined to every route that shares a stop. All of them ask for the fewest moves, so all of them are BFS.

A graph you must deep-copy is walked once, with a map from every old node to its copy. Weighted edges are [Graphs III](#s19); "in what order?" and "are these connected?" are [Graphs II](#s18).

```text
A grid is a graph you never build:              BFS from S spreads in rings of equal distance:

              (r-1, c)                          S . . #            0 1 2 #
    (r, c-1)   (r, c)   (r, c+1)                . # . .    -->     1 # 3 4
              (r+1, c)                          . . . #            2 3 4 #

each cell's edges go to the 4 cells             queue over time:  [0] [1 1] [2 2] [3 3] [4 4]
around it that are on the board                 ring k leaves the queue before ring k+1 starts

DFS on the same grid runs down one corridor as far as it can, then backs up to the last fork.
Same cells, same cost, but its order says nothing about distance.
```

**Why it is fast.** The brute force in most of these problems starts a fresh search from every cell. On a grid of one big island, every search re-walks the whole island, so the work is O((rows · cols)²). One `seen` set shared by all the searches is the whole speed-up.

With a shared `seen`, every node goes into the container at most once, because it is marked the moment it goes in, and every edge is looked at once from each end. A traversal therefore costs O(V + E), which on a grid is O(rows · cols): each cell is pushed once and looks at its four neighbours once.

### From idea to code

**The idea in one sentence:** *put the start in a container and mark it; then repeatedly take one node out, and put in every neighbour you have never seen, marking it as it goes in.*

A queue, oldest out first, makes that BFS. A stack, newest out first, makes it DFS, and that is all a flood fill needs. Only a few jobs need the real DFS tree, with its "on the current path" information: cycle colours, topological order and bridges, all in [Graphs II](#s18). Those use recursion.

The seven decisions for the fewest-steps BFS read as one story. The **state** is a `deque` frontier and a `dist` map, and `dist` doubles as `seen`. By **definition**, `dist[v]` is the fewest moves from the start to `v`, and the queue holds the discovered cells not expanded yet. The **invariant**, true at the end of every step, is that the queue holds at most two distances, the `d`s in front of the `d + 1`s, and that every cell in `dist` is waiting in the queue or already expanded, so it entered exactly once.

A **step** pops a cell and gives every neighbour that is on the board, passable and new the distance `d + 1` as it joins the queue. That is the **record** too: the distance is written at discovery, and it is already final. **Init** puts every source in the queue and in `dist` at distance 0, and the **return** is the target's distance when it is *popped*, or -1 if the queue runs dry.

For counting and measuring regions the story changes little. The state is a `seen` set of the cells already pushed, a stack for the current flood, and two counters: `count`, the floods started so far, and `best`, the largest area. The invariant is that every cell in `seen` is waiting in the stack or already expanded, so it enters the stack exactly once, and that a finished flood has put its whole region into `seen`.

A step pops a cell, then marks and pushes every new land neighbour. The outer loop records `count += 1` when it finds unseen land, and the flood records `best = max(best, area)` when it ends. Everything starts empty with `count = best = 0`, and the return is `count` and `best`.

The sentences you say map onto the lines you type. "The four cells around me" is `for dr, dc in DIRS: nr, nc = r + dr, c + dc`, and "still on the board" is `0 <= nr < rows and 0 <= nc < cols`. "Never discovered" is `(nr, nc) not in seen`, or `not in dist`.

"Mark it as I discover it" is `seen.add(nxt)` on the line just before `queue.append(nxt)`. "The oldest discovery next" is `queue.popleft()`, which makes it BFS; "the newest discovery next" is `stack.pop()`, which makes it DFS. "One ring further out" is `dist[nxt] = dist[cur] + 1`, and "a whole ring at once" is `for _ in range(len(queue)):`. "A new island starts here" is an outer loop over every cell with `if land and not seen: count += 1`.

Turn the story into a graph before you type. Six questions decide every line of a BFS:

1. **Node**: what changes from one move to the next? A cell `(r, c)`, a word, a board string or `(r, c, keys)`; whatever it is goes into `seen`, so it must be hashable, a tuple or a string.
2. **Edge**: what is one legal move? Step onto a free cell, change one letter, turn one wheel; it becomes `def neighbours(node): yield ...`.
3. **Cost**: does every move cost the same? Yes means BFS; only 0 or 1 means a deque, the 0-1 BFS; any w ≥ 0 means a heap, in [Graphs III](#s19).
4. **Start**: one source, or many at once? The entrance, or every rotten orange, or every 0; all of them go into the queue and into `seen` at distance 0.
5. **Goal**: what ends the search? Reaching a cell, holding every key, or running to the end; `if is_goal(node): return d`, checked when the node is *popped*.
6. **Extra**: can two arrivals at the same place have different futures? More keys, more budget left, a different set of visited nodes; then the extra goes into the node, and places × extras must fit in memory, up to about 10⁶ states.

Never put the move count itself in the node: the queue order already carries it, and with it inside, `seen` can no longer merge two arrivals at the same place. On an open 6 × 6 grid, a BFS from corner to corner pops 36 states when the node is the cell, and 111 when it is `(r, c, moves)`.

Step zero is listing a node's neighbours. From an edge list you build an adjacency list once: `graph[u]` holds every node one edge away from `u`, and an undirected edge is written into both lists. For the six nodes and five edges below, node 3 ends up next to 1, 2 and 4.

On a grid you store no edges at all. The neighbours of `(r, c)` are the four cells around it that are still on the board, computed on the fly, and the corner (0, 0) of a 3 × 3 grid has only two of them.

In [ ]:
def build_graph(n, edges, directed=False):
    graph = [[] for _ in range(n)]               # graph[u] = the nodes one edge away from u
    for u, v in edges:
        graph[u].append(v)
        if not directed:
            graph[v].append(u)                   # undirected: store both directions
    return graph


DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))        # down, up, right, left


def grid_neighbours(grid, r, c):
    """The in-bounds cells next to (r, c). A grid is a graph whose edges you never store."""
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            yield nr, nc


print(build_graph(6, [(0, 1), (0, 2), (1, 3), (2, 3), (3, 4)]))   # [[1, 2], [0, 3], [0, 3], [1, 2, 4], [3], []]
print(build_graph(3, [(0, 1), (1, 2)], directed=True))            # [[1], [2], []]
print(list(grid_neighbours(["...", "...", "..."], 0, 0)))         # [(1, 0), (0, 1)]  a corner has 2
print(list(grid_neighbours(["...", "...", "..."], 1, 1)))         # [(2, 1), (0, 1), (1, 2), (1, 0)]

**Try it**
- Compare `build_graph(2, [(0, 1)], directed=True)` with `build_graph(2, [(0, 1)])`: `[[1], []]` versus `[[1], [0]]`. In the directed one, node 1 has no way back to 0.
- Feed 1-indexed edges: `build_graph(3, [(1, 2), (2, 3)])` raises `IndexError` (there is no `graph[3]`). When nodes are numbered from 1, allocate `n + 1` lists.
- An adjacency matrix becomes lists in one line: for `M = [[0, 1, 1], [1, 0, 0], [1, 0, 0]]`, `[[j for j, x in enumerate(row) if x] for row in M]` gives `[[1, 2], [0], [0]]`.

The plain shortest-path question comes first, because every other BFS is this loop with a different node. A maze holds walls `#`, and you want the fewest moves from `start` to `target`, or -1 when no route exists; in the maze below, the corner `S` reaches (1, 3) in 4 moves.

Two moments inside the loop are different on purpose. A cell is marked and given its distance when it is *pushed*, because it was discovered from the closest ring, so that distance is already final. It is compared with the target when it is *popped*, because popping is when it is the closest cell not expanded yet.

In [ ]:
def shortest_steps(grid, start, target):
    """Fewest moves from start to target over cells that are not '#'; -1 if unreachable."""
    start, target = tuple(start), tuple(target)  # LeetCode passes lists; dict keys must be tuples
    rows, cols = len(grid), len(grid[0])
    if grid[start[0]][start[1]] == "#":
        return -1                                 # RETURN: you cannot start inside a wall
    queue = deque([start])                        # STATE + INIT: the frontier, oldest first
    dist = {start: 0}                             # STATE + INIT: dist[cell] = fewest moves; also "seen"
    while queue:
        r, c = queue.popleft()                    # the closest cell not expanded yet
        if (r, c) == target:
            return dist[(r, c)]                   # RETURN: first time it leaves the queue = shortest
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1     # RECORD on discovery: one ring further out, final already
                queue.append((nr, nc))                # STEP: it joins the frontier
    return -1                                     # RETURN: the queue ran dry, target never reached


maze = ["S..#",
        ".#..",
        "...#"]
print(shortest_steps(maze, (0, 0), (1, 3)))            # 4
print(shortest_steps(maze, (0, 0), (2, 2)))            # 4
print(shortest_steps(["S#.", "##."], (0, 0), (1, 2)))  # -1

**Try it**
- Swap `queue.popleft()` for `queue.pop()` (a stack) and run `shortest_steps(["....", "....", "...."], (0, 0), (2, 0))`: it says 8 where BFS says 2. Without the queue's order, "first time out = shortest" stops being true.
- Print `[dist[x] for x in queue]` as the first line inside the loop: the queue never holds more than two distances, `d` and `d + 1`. That is the invariant that makes BFS correct.
- Move the target test to discovery time (return right after the `append` when `(nr, nc) == target`): the same answers, found one ring sooner, except `shortest_steps(["S"], (0, 0), (0, 0))` now returns -1. The start is never "discovered", so it needs its own check.
- Delete the wall-start check: `shortest_steps(["#."], (0, 0), (0, 1))` returns 1, a path that begins inside a wall.

BFS can know a distance in three ways; pick one per problem and never mix them. A `dist` map, `dist = {start: 0}` and then `dist[nxt] = dist[cur] + 1`, gives every node its distance and doubles as `seen`. 01 Matrix, which asks every cell for its distance to the nearest 0, wants exactly that.

The distance can instead ride in the queue, `queue.append((nxt, d + 1))`, which suits a state search that only needs the goal's distance, like the lock below. Or you process whole rings, `for _ in range(len(queue)):` and then `steps += 1`, when the answer is a number of rounds: Rotting Oranges asks how many minutes pass until no fresh orange is left, and one ring is one minute.

A state search is the same loop with a richer node, and Open the Lock shows it in its purest form. A lock has four wheels showing `"0000"`; one move turns one wheel one click up or down; some combinations are dead ends you must never land on; and you want the fewest moves to the target. With the dead ends below, `"0202"` takes 6 moves.

The six answers: the node is the 4-digit string, an edge turns one wheel one click, every move costs 1, the start is `"0000"`, the goal is the target, and nothing extra matters, because a dead end is simply a node you never enter. The loop itself never changes, so it is written once as `bfs_states`, and the six answers go in as arguments.

In [ ]:
def bfs_states(starts, neighbours, is_goal, key=lambda s: s):
    """Fewest moves from any start state to a goal state, or -1. key(state) must be hashable."""
    queue = deque((s, 0) for s in starts)         # STATE + INIT: (state, moves): the distance rides along
    seen = {key(s) for s in starts}               # STATE + INIT: every state ever queued
    while queue:
        state, moves = queue.popleft()
        if is_goal(state):
            return moves                          # RETURN: popped first = fewest moves
        for nxt in neighbours(state):
            if key(nxt) not in seen:
                seen.add(key(nxt))                # STEP: mark on push...
                queue.append((nxt, moves + 1))    # ...one move further out
    return -1                                     # RETURN: every reachable state was tried


def open_lock(deadends, target):
    dead = set(deadends)
    if "0000" in dead:
        return -1
    def turns(s):                                 # one wheel, one click up or down
        for i in range(4):
            for step in (1, -1):
                t = s[:i] + str((int(s[i]) + step) % 10) + s[i + 1:]
                if t not in dead:
                    yield t
    return bfs_states(["0000"], turns, lambda s: s == target)


print(open_lock(["0201", "0101", "0102", "1212", "2002"], "0202"))   # 6
print(open_lock(["8888"], "0009"))                                   # 1
print(open_lock(["8887", "8889", "8878", "8898", "8788", "8988", "7888", "9888"], "8888"))   # -1

**Try it**
- Delete the `if "0000" in dead` check: `open_lock(["0000"], "8888")` answers 8 instead of -1. `turns` only filters the states you move *to*, so the start itself needs its own check.
- Turn the wheels upward only (`for step in (1,)`): `open_lock([], "0009")` takes 9 moves instead of 1. Each edge you leave out is a move the search can never make.
- Predict `open_lock(["0001", "0009", "0010", "0090", "0100", "0900", "1000", "9000"], "0002")` before running it: -1, because every first move is a dead end.

The DFS template answers the other big question, how many regions there are and how big. Number of Islands counts the groups of `"1"` cells joined up, down, left or right; the sea below has 3 islands. Max Area of Island asks for the biggest one, 4 cells here.

The outer loop is what turns "flood one region" into "count the regions": every cell gets a chance to start a flood, and only unseen land does. A cell is marked when it is pushed, so it can never be pushed twice, and it is counted when it is popped. The recursive version does the same job on the call stack: it checks a cell when it arrives there, then visits the four neighbours.

In [ ]:
def islands(grid):
    """(number of islands, largest island area) for a grid of '1' land and '0' water."""
    rows, cols = len(grid), len(grid[0])
    seen = set()                                  # STATE + INIT: cells already pushed (never push twice)
    count = best = 0                              # STATE + INIT: islands so far, largest area so far
    for r in range(rows):
        for c in range(cols):                     # every cell gets a chance to start a flood
            if grid[r][c] != "1" or (r, c) in seen:
                continue
            count += 1                            # RECORD: unseen land = a brand-new island
            seen.add((r, c))
            stack, area = [(r, c)], 0
            while stack:                          # flood the island with an explicit stack
                x, y = stack.pop()
                area += 1                         # every cell is popped exactly once
                for dx, dy in DIRS:
                    nx, ny = x + dx, y + dy
                    if 0 <= nx < rows and 0 <= ny < cols and grid[nx][ny] == "1" and (nx, ny) not in seen:
                        seen.add((nx, ny))        # STEP: mark on push...
                        stack.append((nx, ny))    # ...so no cell is pushed twice
            best = max(best, area)                # RECORD: the island is complete
    return count, best                            # RETURN


def area_recursive(grid, r, c, seen):
    """The same flood, recursive: check when you ARRIVE, then visit the 4 neighbours."""
    if not (0 <= r < len(grid) and 0 <= c < len(grid[0])) or grid[r][c] != "1" or (r, c) in seen:
        return 0                                  # off the board, water, or counted already
    seen.add((r, c))
    return 1 + sum(area_recursive(grid, r + dr, c + dc, seen) for dr, dc in DIRS)


sea = ["11000",
       "11000",
       "00100",
       "00011"]
print(islands(sea))                       # (3, 4)
print(area_recursive(sea, 0, 0, set()))   # 4

**Try it**
- Mark at pop time instead: delete `seen.add((nx, ny))` and put `seen.add((x, y))` right after `x, y = stack.pop()`. `islands(["11", "11"])` now says `(1, 5)`: cell (1, 0) is pushed by (0, 0) and again by (1, 1), so it is popped twice.
- Delete `0 <=` from `0 <= nx < rows` and run `islands(["1", "0", "1"])`: `(2, 2)`. The top cell's "up" neighbour is `grid[-1]`, the bottom row, so the first island counts a phantom second cell.
- Run `area_recursive(["1" * 5000], 0, 0, set())`: `RecursionError`. The iterative `islands(["1" * 5000])` answers `(1, 5000)` without trouble.
- Collect every island's `area` in a list instead of keeping the max: `sea` gives `[1, 2, 4]` once sorted.

### Watch it work

The rings are the whole argument for BFS, so watch them form. Each printed line is one ring of the search through the maze above, and at the end the grid shows every cell's distance from `S`, with `?` for a cell the wave never reached.

In [ ]:
def trace_rings(grid, start):
    dist, queue, ring = {start: 0}, deque([start]), 0
    while queue:
        layer = [queue.popleft() for _ in range(len(queue))]   # exactly the cells at distance `ring`
        print(f"ring {ring}: {layer}")
        for r, c in layer:
            for nr, nc in grid_neighbours(grid, r, c):
                if grid[nr][nc] != "#" and (nr, nc) not in dist:
                    dist[(nr, nc)] = ring + 1
                    queue.append((nr, nc))
        ring += 1
    for r, row in enumerate(grid):
        print("   ", " ".join("#" if ch == "#" else str(dist.get((r, c), "?")) for c, ch in enumerate(row)))


trace_rings(maze, (0, 0))

**Try it**
- Predict the distances for `trace_rings(maze, (1, 2))`, then run it: the rings spread both ways and the far cell (1, 0) is 4 away.
- Wall in a cell: `trace_rings(["S..#", ".##.", "...#"], (0, 0))`. Cell (1, 3) keeps its `?`: that is the `-1` case of `shortest_steps`.
- Pop one cell per iteration instead of a whole layer, write `dist[(r, c)] + 1` instead of `ring + 1`, and print each popped cell's distance: `0 1 1 2 2 3 3 4 4`, never decreasing.

### Where it goes wrong

1. **Marking `seen` when you pop instead of when you push.** Between its first push and its pop, a node is pushed again by every other neighbour that reaches it, so the queue fills with copies, and anything counted per pop is counted more than once: a 2×2 block of land gets area 5. The other correct form is to mark on pop *and skip a node that is already marked*: copies still enter the queue, but none is expanded twice. Dijkstra must use that form ([Graphs III](#s19)).
2. **Half a bounds check.** Without `0 <=`, Python reads `grid[-1]` as the last row instead of failing, so the top edge silently touches the bottom edge: `islands(["1", "0", "1"])` reports `(2, 2)` instead of `(2, 1)`. Always write `0 <= nr < rows`.
3. **Forgetting to mark the start.** Without `seen.add((r, c))` before the flood, the start is pushed again by its own neighbour: `islands(["11"])` returns `(1, 3)`.
4. **One search on a disconnected graph.** With `build_graph(4, [(0, 1), (2, 3)])`, a search from 0 reaches only {0, 1}. Counting or visiting everything needs the outer loop that tries every node as a start.
5. **DFS for "fewest steps".** DFS finds *a* path, not the shortest: on an open 3×4 grid a stack-based search reports 8 moves from (0, 0) to (2, 0), BFS reports 2.
6. **Counting pops instead of rings.** When the answer is a number of rounds, drop the ring loop (or put `minutes += 1` inside it) and you count pops: the first Rotting Oranges grid answers 6 instead of 4. Finish the whole ring, then add one.
7. **Deep recursion.** A recursive flood of a 300×300 island can need 90,000 nested calls; Python stops at about 1,000. Use an explicit stack (or BFS) on big grids, and say so.
8. **Flood fill with the new colour equal to the old one.** A fill that uses the paint as its "seen" mark never ends on `[[1, 1]]` from (0, 0) with colour 1: painting changes nothing, so the two cells keep pushing each other. Return early when the colours are equal.
9. **An undirected edge stored once.** `build_graph(2, [(0, 1)], directed=True)[1]` is `[]`: from node 1 there is no way back.
10. **Positions given as lists.** LeetCode passes `[0, 0]`: `shortest_steps(maze, [0, 0], (1, 3))` without the conversion raises `TypeError: unhashable type: 'list'`, and a list `target` silently returns -1 because `(1, 3) == [1, 3]` is False. Convert first: `start, target = tuple(start), tuple(target)`.
11. **`1` versus `"1"`.** Number of Islands stores the strings `"1"`/`"0"`; Max Area of Island and Number of Enclaves, which counts the land cells that cannot walk off the grid, store the ints. `islands([[1, 1], [0, 1]])` returns `(0, 0)`, because `1 != "1"`.
12. **`seen` keyed on less than the whole state.** In BFS over states, `(cell, keys)` is the node, not `cell`. Shortest Path to Get All Keys asks for the fewest moves to collect every key when each lock needs its key; with `seen` keyed on the cell, `["a.@A.b"]` answers -1 instead of 7, because the walk must come back over cells it has already crossed.

### Edge cases to say out loud

Start equals target · start or target is a wall · target unreachable · no land at all · one big island · diagonal cells (not neighbours unless the problem says 8 directions) · nodes with no edges · nodes numbered from 1 · positions given as lists.

In [ ]:
assert shortest_steps(["S"], (0, 0), (0, 0)) == 0            # the start is the target
assert shortest_steps(["#."], (0, 0), (0, 1)) == -1          # you cannot start inside a wall
assert shortest_steps(["S#."], (0, 0), (0, 2)) == -1         # a wall cuts the only way
assert shortest_steps(maze, [0, 0], [1, 3]) == 4             # positions given as lists still work
assert islands(["000"]) == (0, 0)                             # no land: 0 islands, area 0
assert islands(["111", "111"]) == (1, 6)                      # one island covers everything
assert islands(["1", "0", "1"]) == (2, 1)                     # top and bottom rows do not touch
assert islands(["101", "010", "101"]) == (5, 1)               # diagonal cells are not neighbours
assert build_graph(3, []) == [[], [], []]                     # nodes without edges still exist
print("edge cases pass")

**Try it**
- Predict, then add: `assert shortest_steps(["S.", ".."], (0, 0), (1, 1)) == 2`.
- What should `islands(["1"])` return? Write the assert before running it (`(1, 1)`).
- Try 8 directions without touching the shared `DIRS`: copy `islands` as `islands8`, define `DIRS8 = DIRS + ((1, 1), (1, -1), (-1, 1), (-1, -1))` and loop over `DIRS8` inside it. `islands8(["101", "010", "101"])` is one island of 5.

### Variations

Every variation below keeps the loop and changes one thing: where the search starts, what a node is, or what rides along with it. The table is the overview; each variation then gets its own paragraph and, for the important ones, its code.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Count / measure regions** | outer loop over every cell; count the launches; the flood returns its size | Number of Islands (200), Max Area of Island (695), Flood Fill (733) |
| **Multi-source BFS** | push *every* source before the loop; a ring = one minute, or one step from the nearest source | Rotting Oranges (994), Walls and Gates (286), 01 Matrix (542) |
| **Return the path** | a `parent` map instead of `dist`; walk it back from the target | any shortest path |
| **8 directions, blocked start** | 8 entries in the direction list; check the start cell before the loop | Shortest Path in Binary Matrix (1091) |
| **Border-first** | flood from the border instead of asking every cell; what stays unmarked is enclosed | Surrounded Regions (130), Number of Enclaves (1020), Pacific Atlantic Water Flow (417) |
| **Two floods** | flood one island to collect it, then multi-source BFS from all its cells toward the other | Shortest Bridge (934) |
| **Copy a graph** | an `old -> new` dict is both `seen` and the lookup for neighbours | Clone Graph (133) |
| **Carry a label across edges** | a colour flips (bipartite) or a ratio multiplies (division) along each edge | Is Graph Bipartite (785), Possible Bipartition (886), Evaluate Division (399) |
| **BFS over states** | the node is a string or a tuple; neighbours come from a function; `bfs_states` is unchanged | Open the Lock (752), Word Ladder (127) |
| **BFS over augmented states** | node = (cell, extra); `seen` keys on the whole tuple | Shortest Path in a Grid with Obstacles Elimination (1293) |
| *Second pass:* **richer nodes** | the node is a whole board, (cell, keys held) or (node, visited mask) | Sliding Puzzle (773), Shortest Path to Get All Keys (864), Shortest Path Visiting All Nodes (847) |
| *Second pass:* **BFS over groups** | expand a whole group (a bus route, all equal values) once, then never again | Bus Routes (815), Jump Game IV (1345) |
| *Second pass:* **pruned branching** | branch only on moves that fix the first mismatch | K-Similar Strings (854) |
| *Second pass:* **every shortest path** | keep every parent from the previous ring; walk back from the end | Word Ladder II (126) |

Flood Fill is the smallest member of the first row: recolour the start pixel and every pixel of the same colour connected to it, so `[[1, 1, 1], [1, 1, 0], [1, 0, 1]]` painted with 2 from the middle becomes `[[2, 2, 2], [2, 2, 0], [2, 0, 1]]`. The paint itself serves as the visited mark, which is why trap 8 exists.

Shortest Path in Binary Matrix is the template with 8 directions and a start cell that may itself be blocked: the fewest cells on a path of 0s from the top-left to the bottom-right corner, moving in any of 8 directions, or -1.

The multi-source variation comes first because it changes only the first line of the template. Rotting Oranges is worked line by line in [From Idea to Code](01_Start_Here.ipynb#s01). The same idea with distances instead of minutes is 01 Matrix: for every cell, how far is the nearest 0, so `[[0, 1, 1, 1, 0]]` becomes `[[0, 1, 2, 1, 0]]`.

Every 0 is a source, so all of them go into the queue at distance 0 before the loop starts, and the first time the wave reaches a cell it came from the *nearest* 0. Walls and Gates asks for the distance from every empty room to its nearest gate, with walls in the way; it is the same code with the gates as the zeros, and a room the wave never reaches keeps its starting value.

In [ ]:
def update_matrix(mat):
    rows, cols = len(mat), len(mat[0])
    dist = [[-1] * cols for _ in range(rows)]       # STATE: -1 = not reached yet, so dist doubles as seen
    queue = deque()
    for r in range(rows):
        for c in range(cols):
            if mat[r][c] == 0:
                dist[r][c] = 0                      # INIT: every source at distance 0
                queue.append((r, c))
    while queue:
        r, c = queue.popleft()
        for nr, nc in grid_neighbours(mat, r, c):
            if dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1       # RECORD on discovery: already final
                queue.append((nr, nc))              # STEP
    return dist                                     # RETURN


print(update_matrix([[0, 0, 0], [0, 1, 0], [1, 1, 1]]))   # [[0, 0, 0], [0, 1, 0], [1, 2, 1]]
print(update_matrix([[0, 1, 1, 1, 0]]))                   # [[0, 1, 2, 1, 0]]

**Try it**
- Seed only the first 0: `update_matrix([[0, 1, 1, 1, 0]])` gives `[[0, 1, 2, 3, 4]]` instead of `[[0, 1, 2, 1, 0]]`. Every source has to start at the same moment.
- Swap `popleft()` for `pop()`: `[[0, 1, 1, 1, 0]]` comes out `[[0, 3, 2, 1, 0]]`. A stack hands out the newest cell, so a first visit is no longer the closest one.
- Count the pushes on a 30 × 30 grid with zeros on the diagonal: 900 with every zero as a source at once, 27,000 if you run one BFS per zero and keep the minimum.
- Print `list(queue)` right after the seeding loops for the first matrix: all five zeros wait at distance 0, as if one invisible super-source, joined to every 0 by a free edge, had just been expanded. Multi-source BFS is a plain BFS from that super-source.

Interviewers often follow the distance with "now show me the path", so the parent map comes next. Remember who discovered each cell in `parent`, and walk those links back from the target; in the maze the route from `S` to (1, 3) runs along the top row, then down and one step right, five cells in all. The `parent` map also serves as `seen`, because a cell has a parent exactly when it has been discovered.

In [ ]:
def shortest_path(grid, start, target):
    parent = {start: None}                  # STATE + INIT: parent[cell] = the cell that discovered it; also "seen"
    queue = deque([start])
    while queue:
        cur = queue.popleft()
        if cur == target:                   # RECORD: walk the parents back to the start
            path = []
            while cur is not None:
                path.append(cur)
                cur = parent[cur]
            return path[::-1]               # RETURN
        for nxt in grid_neighbours(grid, *cur):
            if grid[nxt[0]][nxt[1]] != "#" and nxt not in parent:
                parent[nxt] = cur           # STEP: remember who discovered nxt
                queue.append(nxt)
    return []


print(shortest_path(maze, (0, 0), (1, 3)))             # [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3)]
print(shortest_path(["S#.", "##."], (0, 0), (1, 2)))   # []

**Try it**
- With `pop()` instead of `popleft()`, the open 3×4 grid `["....", "....", "...."]` from (0, 0) to (2, 0) returns an 8-move snake instead of the 2-move path.
- `len(shortest_path(maze, (0, 0), (1, 3))) - 1` is 4, the same number `shortest_steps` gives: a path of k + 1 cells has k moves.
- `shortest_path(maze, (0, 0), (0, 0))` is `[(0, 0)]`: a path with zero moves.

The next two variations change where the flood starts. Surrounded Regions asks you to flip every region of `O`s that does not touch the border, so the board `["XXXX", "XOOX", "XXOX", "XOXX"]` keeps only its bottom `O`.

"Can this `O` reach the border?" asked for every cell repeats the same search. Asked the other way round, *which cells can the border reach?*, it is one flood from all border cells, and whatever stays unmarked is enclosed. Number of Enclaves asks the same question for land cells, counting the ones that cannot walk off the grid.

Pacific Atlantic Water Flow asks which cells can send water to both oceans, with the Pacific along the top and left edges and the Atlantic along the bottom and right, when water only flows to a neighbour that is equally high or lower. On `[[1, 2], [4, 3]]` three cells manage it: (0, 1), (1, 0) and (1, 1).

From each ocean we walk *uphill*, and the cells both floods reach are the answer. The helper takes the step rule as a function, because that rule is the only thing that changes between the two problems.

In [ ]:
def flood(grid, starts, can_step):
    """Every cell reachable from any start, stepping from a to b only when can_step(a, b)."""
    seen, stack = set(starts), list(starts)
    while stack:
        r, c = stack.pop()
        for nr, nc in grid_neighbours(grid, r, c):
            if (nr, nc) not in seen and can_step((r, c), (nr, nc)):
                seen.add((nr, nc))
                stack.append((nr, nc))
    return seen


def capture_surrounded(board):
    rows, cols = len(board), len(board[0])
    edge_o = [(r, c) for r in range(rows) for c in range(cols)
              if board[r][c] == "O" and (r in (0, rows - 1) or c in (0, cols - 1))]
    safe = flood(board, edge_o, lambda a, b: board[b[0]][b[1]] == "O")   # O's that touch the border
    return ["".join("O" if (r, c) in safe else "X" for c in range(cols)) for r in range(rows)]


def pacific_atlantic(h):
    rows, cols = len(h), len(h[0])
    uphill = lambda a, b: h[b[0]][b[1]] >= h[a[0]][a[1]]                # water flows down, so we walk up
    pacific = flood(h, [(0, c) for c in range(cols)] + [(r, 0) for r in range(rows)], uphill)
    atlantic = flood(h, [(rows - 1, c) for c in range(cols)] + [(r, cols - 1) for r in range(rows)], uphill)
    return sorted(pacific & atlantic)


print(capture_surrounded(["XXXX", "XOOX", "XXOX", "XOXX"]))   # ['XXXX', 'XXXX', 'XXXX', 'XOXX']
heights = [[1, 2, 2, 3, 5],
           [3, 2, 3, 4, 4],
           [2, 4, 5, 3, 1],
           [6, 7, 1, 4, 5],
           [5, 1, 1, 2, 4]]
print(pacific_atlantic(heights))   # [(0, 4), (1, 3), (1, 4), (2, 2), (3, 0), (3, 1), (4, 0)]

**Try it**
- Change `>=` to `>` in `uphill`: `(1, 4)` disappears from the answer. Its only way to the Pacific crosses `(1, 3)`, which has the same height.
- Print `len(pacific), len(atlantic)` inside `pacific_atlantic`: 16 and 16, with 7 cells in common.
- Number of Enclaves with the same helper: for `g = [[0, 0, 0, 0], [1, 0, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]]`, flood from the border land cells with `lambda a, b: g[b[0]][b[1]] == 1`, then subtract the number reached from the total land: 3.
- `capture_surrounded(["XOX", "XOX", "XXX"])` captures nothing: that region touches the top edge.

Shortest Bridge joins the two ideas above. A grid holds exactly two islands, and you want the fewest water cells to flip so that they touch. Flood one island with the DFS template to collect its cells, then run a multi-source BFS from all of them at once; the number of water rings the wave crosses before it first touches the other island is the answer.

Copying a graph is a traversal too, and the only new question is what `seen` should be. Clone Graph hands you one node of an undirected graph, each node holding a value and a neighbour list, and asks for a deep copy: the square 1-2-3-4-1 below must come back as four fresh nodes wired into the same square.

The copy needs one new node per old node, and every edge must connect *copies*. A dict `old -> new` answers "do I already have a copy of this node?" in O(1), which also makes it the visited set, and that is what stops the traversal from going round the cycle forever.

In [ ]:
class Node:
    def __init__(self, val):
        self.val = val
        self.neighbors = []                       # LeetCode's attribute name, American spelling


def clone_graph(node):
    if node is None:
        return None
    clones = {node: Node(node.val)}               # old -> new: the visited set AND the lookup table
    queue = deque([node])
    while queue:
        cur = queue.popleft()
        for nb in cur.neighbors:
            if nb not in clones:                  # first sighting: make its copy, expand it later
                clones[nb] = Node(nb.val)
                queue.append(nb)
            clones[cur].neighbors.append(clones[nb])   # wire copy -> copy, never copy -> original
    return clones[node]


a, b, c, d = Node(1), Node(2), Node(3), Node(4)             # the square 1-2-3-4-1
a.neighbors, b.neighbors, c.neighbors, d.neighbors = [b, d], [a, c], [b, d], [a, c]
clone = clone_graph(a)
print(clone.val, [n.val for n in clone.neighbors])         # 1 [2, 4]
print(clone is a, clone.neighbors[0] is b)                 # False False
print(clone.neighbors[0].neighbors[0] is clone)            # True  (the cycle closes on the copy)

**Try it**
- In the append line, use `nb` instead of `clones[nb]`: now `clone.neighbors[0] is b` prints True. The copy points back into the original graph.
- Delete `queue.append(nb)`: the last line now raises `IndexError`, because `clone.neighbors[0].neighbors` is empty. The copies get made, but only the start is ever expanded.
- Print `len(clones)` right before the `return`: 4 copies, although every node was reached twice.
- `clone_graph(None)` is `None`, and a lone `Node(7)` copies to a node with no neighbours.

Sometimes a node receives a value from the node that discovered it, and the traversal carries that label along every edge. Is Graph Bipartite asks whether the nodes can be split into two camps so that every edge joins the camps; Possible Bipartition is the same question with "dislikes" as the edges.

Here the label is a colour that *flips* across every edge. An edge whose two ends got the same colour closes an odd cycle, so the answer is False.

Evaluate Division gives facts like `a / b = 2.0` and `b / c = 3.0` and asks for ratios like `a / c`, which is 6.0. The label is a ratio that *multiplies* along the path: `a / b = 2` is an edge a → b worth 2 and an edge b → a worth 1/2.

In [ ]:
def is_bipartite(graph):
    color = {}                                # STATE: color[v] = 0 or 1; also "seen"
    for s in range(len(graph)):               # every component needs its own start
        if s in color:
            continue
        color[s] = 0                          # INIT
        queue = deque([s])
        while queue:
            u = queue.popleft()
            for v in graph[u]:
                if v not in color:
                    color[v] = 1 - color[u]   # STEP: neighbours get the other colour
                    queue.append(v)
                elif color[v] == color[u]:
                    return False              # RECORD: an edge inside one colour = an odd cycle
    return True                               # RETURN


def calc_equation(equations, values, queries):
    graph = defaultdict(list)                 # a / b = k: edge a -> b worth k, edge b -> a worth 1 / k
    for (x, y), k in zip(equations, values):
        graph[x].append((y, k))
        graph[y].append((x, 1 / k))

    def ratio(src, dst):                      # DFS that carries src / node along the path
        if src not in graph or dst not in graph:
            return -1.0
        seen, stack = {src}, [(src, 1.0)]
        while stack:
            node, r = stack.pop()
            if node == dst:
                return r
            for nxt, k in graph[node]:
                if nxt not in seen:
                    seen.add(nxt)
                    stack.append((nxt, r * k))    # src / nxt = (src / node) * (node / nxt)
        return -1.0

    return [ratio(x, y) for x, y in queries]


print(is_bipartite([[1, 3], [0, 2], [1, 3], [0, 2]]), is_bipartite([[1, 2], [0, 2], [0, 1]]))   # True False
print(is_bipartite([[1], [0], [3, 4], [2, 4], [2, 3]]))                                         # False
print(calc_equation([["a", "b"], ["b", "c"]], [2.0, 3.0],
                    [["a", "c"], ["b", "a"], ["a", "e"], ["a", "a"], ["x", "x"]]))             # [6.0, 0.5, -1.0, 1.0, -1.0]

**Try it**
- Start from node 0 only (drop the outer loop): the disconnected example `[[1], [0], [3, 4], [2, 4], [2, 3]]` now says True. The triangle 2-3-4 is never coloured.
- Print `color` at the end for the square `[[1, 3], [0, 2], [1, 3], [0, 2]]`: `{0: 0, 1: 1, 3: 1, 2: 0}`. Opposite corners share a colour.
- Forget the reverse edge (delete the `graph[y].append(...)` line): `b / a` comes back -1.0 instead of 0.5.
- Ask for `["c", "a"]`: 0.16666666666666666, found by walking c → b → a and multiplying 1/3 · 1/2.

Open the Lock already showed BFS over states, and Word Ladder has exactly its shape. It changes one letter at a time through a word list and asks how many words the shortest chain from `"hit"` to `"cog"` has: 5, along hit, hot, dot, dog, cog. A word's neighbours come from buckets like `"h*t"`, one per word and blanked position, instead of comparing every pair of words, and `bfs_states` does the rest. Without `"cog"` in the list no ladder exists, and the answer is 0.

In [ ]:
def ladder_length(begin, end, words):
    buckets = defaultdict(list)                   # "h*t" -> every word that fits the pattern
    for w in words:
        for i in range(len(w)):
            buckets[w[:i] + "*" + w[i + 1:]].append(w)

    def one_letter_away(w):
        for i in range(len(w)):
            yield from buckets[w[:i] + "*" + w[i + 1:]]

    moves = bfs_states([begin], one_letter_away, lambda w: w == end)
    return moves + 1 if moves != -1 else 0        # the problem counts words, not moves


print(ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log", "cog"]))     # 5
print(ladder_length("hit", "cog", ["hot", "dot", "dog", "lot", "log"]))            # 0

**Try it**
- Add `print(w, end=" ")` as the first line of `one_letter_away` and rerun the cell: the first line now starts `hit hot dot lot dog log`, the order in which BFS expands the words, ring by ring.
- Return `moves` instead of `moves + 1`: the first ladder says 4. A chain of 5 words has 4 moves, and the problem counts words.
- Print `buckets["*ot"]` inside `ladder_length`: `['hot', 'dot', 'lot']`. Every pair in one bucket is one letter apart, so the bucket lists those neighbours without comparing a single pair of words.
- Put `"hit"` itself into the first word list and rerun: still 5. The start's own buckets now hand `"hit"` back, and `seen` already holds it, so nothing changes.

When the same cell can be in different situations, the situation becomes part of the node: question 6 of the recipe. Shortest Path in a Grid with Obstacles Elimination lets you break at most k walls between the top-left and bottom-right corners and asks for the fewest steps: the 5 × 3 grid below takes 6 with k = 1, and -1 comes back when k is too small. The node is (cell, eliminations left), and since every move still costs one step, `bfs_states` works unchanged, with `seen` keyed on the whole tuple.

In [ ]:
def eliminate_obstacles(grid, k):
    rows, cols = len(grid), len(grid[0])

    def moves(state):
        r, c, left = state
        for nr, nc in grid_neighbours(grid, r, c):
            if left - grid[nr][nc] >= 0:          # stepping onto a 1 spends one elimination
                yield nr, nc, left - grid[nr][nc]
    return bfs_states([(0, 0, k)], moves, lambda s: (s[0], s[1]) == (rows - 1, cols - 1))


print(eliminate_obstacles([[0, 0, 0], [1, 1, 0], [0, 0, 0], [0, 1, 1], [0, 0, 0]], 1))   # 6
print(eliminate_obstacles([[0, 1, 1], [1, 1, 1], [1, 0, 0]], 1))                        # -1

**Try it**
- Run the first grid with `k = 0`: 10, the long way round. One elimination already buys the straight route, so any k ≥ 1 gives 6.
- Pass `key=lambda s: s[:2]` to `bfs_states`, so `seen` remembers only the cell, and run `eliminate_obstacles([[0, 0, 1, 1], [1, 0, 1, 0]], 1)`: -1 instead of 4. Cell (1, 1) is first reached through the wall at (1, 0), with the budget spent, and that arrival blocks the one through (0, 1) that still has it.
- Before running, bound the work: R · C cells times k + 1 budgets, 5 · 3 · 2 = 30 states for the first grid. Then print `len(seen)` in `bfs_states` just before `return moves` and rerun both cells: the first grid queued 22 states.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Three of them only need a richer node for `bfs_states`. Sliding Puzzle holds the tiles 1 to 5 and one blank on a 2 × 3 board; one move slides a tile into the blank, and it asks for the fewest moves to reach `123` over `450`, or -1 when the board can never get there: `[[1, 2, 3], [4, 0, 5]]` needs 1 move. The node is the whole board written as a string, which is hashable, so it fits in `seen`.

Shortest Path to Get All Keys has a start `@`, walls `#`, keys as lowercase letters and locks as the matching uppercase letters, and asks for the fewest moves to collect every key: `["@.a..", "###.#", "b.A.B"]` takes 8. A lock can only be crossed with its key in hand, so the node is (cell, keys held), with the keys as a bitmask.

Shortest Path Visiting All Nodes asks for the shortest walk that visits every node of a small graph, starting anywhere and revisiting freely: the star `[[1, 2, 3], [0], [0], [0]]` takes 4 steps. The node is (node, visited mask), and every node is a start, so all of them enter the queue at distance 0.

In [ ]:
TOUCH = {0: (1, 3), 1: (0, 2, 4), 2: (1, 5), 3: (0, 4), 4: (1, 3, 5), 5: (2, 4)}   # 2x3 board, by index


def sliding_puzzle(board):
    def slides(s):                                # swap the blank with each tile touching it
        z = s.index("0")
        for j in TOUCH[z]:
            t = list(s)
            t[z], t[j] = t[j], t[z]
            yield "".join(t)
    start = "".join(str(x) for row in board for x in row)       # a board becomes a hashable string
    return bfs_states([start], slides, lambda s: s == "123450")


def all_keys(grid):
    rows, cols = len(grid), len(grid[0])
    start = next((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == "@")
    full = (1 << sum(ch.islower() for row in grid for ch in row)) - 1   # one bit per key

    def moves(state):
        r, c, keys = state
        for nr, nc in grid_neighbours(grid, r, c):
            ch = grid[nr][nc]
            bit = 1 << (ord(ch.lower()) - ord("a")) if ch.isalpha() else 0   # key a and lock A share a bit
            if ch == "#" or (ch.isupper() and not keys & bit):
                continue                          # a wall, or a lock whose key we do not hold
            yield nr, nc, (keys | bit if ch.islower() else keys)
    return bfs_states([(*start, 0)], moves, lambda s: s[2] == full)


def visit_all_nodes(graph):
    full = (1 << len(graph)) - 1
    starts = [(i, 1 << i) for i in range(len(graph))]     # start anywhere: every node is a source
    walk = lambda s: ((v, s[1] | (1 << v)) for v in graph[s[0]])
    return bfs_states(starts, walk, lambda s: s[1] == full)


print(sliding_puzzle([[1, 2, 3], [4, 0, 5]]), sliding_puzzle([[4, 1, 2], [5, 0, 3]]),
      sliding_puzzle([[1, 2, 3], [5, 4, 0]]))                                        # 1 5 -1
print(all_keys(["@.a..", "###.#", "b.A.B"]), all_keys(["@..aA", "..B#.", "....b"]), all_keys(["@Aa"]))   # 8 6 -1
print(visit_all_nodes([[1, 2, 3], [0], [0], [0]]))                                 # 4

**Try it**
- In `bfs_states` (in the Open the Lock cell), print `len(seen)` just before `return -1` and rerun that cell; then call only `sliding_puzzle([[1, 2, 3], [5, 4, 0]])`: 360. Only half of the 720 boards can be reached from it.
- Predict `sliding_puzzle([[3, 2, 4], [1, 5, 0]])` before running it (14).
- In `all_keys`, pass `key=lambda s: s[:2]` to `bfs_states`, so `seen` remembers only the cell, and run `all_keys(["a.@A.b"])`: -1 instead of 7. After fetching `a` on the left, the walk must cross cells it has already seen to reach the lock.
- In `visit_all_nodes`, start from node 0 only (`starts = [(0, 1)]`): 5 instead of 4. Starting at a leaf is shorter, which is why every node is a source.

Sometimes one move reaches a whole group at once, every stop of a bus route or every index holding the same value. The group is the expensive part, so each group is expanded *once*.

Bus Routes gives each bus as the loop of stops it drives, and asks for the fewest buses from a source stop to a target stop; with routes `[1, 2, 7]` and `[3, 6, 7]`, stop 1 reaches stop 6 on 2 buses. The problem counts buses, so the nodes are routes, reached through a stop → routes index.

In [ ]:
def num_buses(routes, source, target):
    if source == target:
        return 0
    routes_at = defaultdict(list)                 # stop -> every route through it
    for i, route in enumerate(routes):
        for stop in route:
            routes_at[stop].append(i)
    queue = deque((i, 1) for i in routes_at[source])   # boarding any bus at source = 1 bus
    boarded, expanded = set(routes_at[source]), {source}
    while queue:
        i, buses = queue.popleft()
        for stop in routes[i]:
            if stop == target:
                return buses
            if stop in expanded:
                continue
            expanded.add(stop)                    # look at each stop's routes once
            for j in routes_at[stop]:
                if j not in boarded:
                    boarded.add(j)                # board each route once
                    queue.append((j, buses + 1))
    return -1


print(num_buses([[1, 2, 7], [3, 6, 7]], 1, 6))                                 # 2
print(num_buses([[7, 12], [4, 5, 15], [6], [15, 19], [9, 12, 13]], 15, 12))    # -1

**Try it**
- Delete `if source == target: return 0`: `num_buses([[1, 2, 7], [3, 6, 7]], 1, 1)` returns 1, a bus ride you never needed.
- Print `i, buses` after each `popleft()` for the first example: route 0 with 1 bus, then route 1 with 2 buses. Stops are never counted.
- `num_buses([[1, 2, 3, 4, 5, 6]], 1, 6)` is 1: one ride covers five stops, because an edge is a bus, not a stop.

Jump Game IV is the same idea on an array. From index i you may jump to i − 1, i + 1 or any index holding the same value, and it asks for the fewest jumps from the first index to the last: 3 for `[100, -23, -23, 404, 100, 23, 23, 23, 3, 404]`. Run BFS over indexes and delete a value's bucket of equal indexes the first time it is used, so a long run of equal values is read once instead of on every pop.

K-Similar Strings asks for the fewest swaps of two letters that turn one anagram into another: `"abac"` becomes `"baca"` in 2. It is BFS over strings that branches only on swaps putting the right letter into the first wrong slot, and some shortest sequence always begins with such a swap, so the pruning loses nothing.

Word Ladder II asks for *every* shortest ladder, not just its length, so the ring structure has to be kept. Run the BFS ring by ring and record, for every word, *all* the words of the previous ring that reach it, a parents map.

Remove a ring's words from the dictionary only after the whole ring is done, so that two parents in the same ring can both register. Stop after the ring that contains `endWord`, then walk the parents map backwards from `endWord` to list every path.

### Say it in the interview

Regions:

> "Each land cell is a node joined to its land neighbours, so an island is a connected component. A fresh search from every cell re-walks each island, O((mn)²); with one shared `seen` every cell is pushed once, O(mn). Unseen land launches a new flood: one more island. I flood with an explicit stack so a big island can't hit the recursion limit."

Fewest moves:

> "Every move costs 1, so BFS. A cell at distance k + 1 can only be discovered from a cell at distance k, and ring k leaves the queue before ring k + 1, so the first time the target is popped its distance is minimal. I mark on enqueue, so each cell enters once: O(mn)."

Point at the `seen.add` right before the `append`, the bounds check with both halves, and the line where you record. For a state search, answer the six questions out loud before writing any code.

Expect follow-ups, and have the one-line answer ready. The path itself is a parent map. Eight directions is a longer direction list. "Don't modify the input" is a `seen` set instead of sinking cells. Recursion depth is an explicit stack. A Word Ladder that is too slow is bidirectional BFS: search from both ends and always expand the smaller frontier.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| 01 Matrix | `graphs/zero_one_matrix.py` | multi-source BFS from every 0; a cell's first visit is its distance |
| Bus Routes | `graphs/bus_routes.py` | BFS over routes, not stops: a stop → routes index; board each route once, expand each stop once |
| Clone Graph | `graphs/clone_graph.py` · `practice/simple/42_clone_graph.py` | one traversal; the old → new dict is both the visited set and the neighbour lookup |
| Flood Fill | `graphs/flood_fill.py` | recolouring is the visited mark; return early if the new colour equals the old one |
| Jump Game IV | `graphs/jump_game_iv.py` | BFS on indexes; a value's bucket of equal indexes is used once, then deleted |
| K-Similar Strings | `graphs/k_similar_strings.py` | BFS over strings; branch only on swaps that fix the first mismatched slot |
| Max Area of Island | `graphs/max_area_of_island.py` | flood each island once from its first cell; the flood returns its size |
| Number of Enclaves | `graphs/number_of_enclaves.py` | flood the land the border can reach; the land left over is enclosed |
| Number of Islands | `graphs/number_of_islands.py` · `practice/simple/41_number_of_islands.py` | count how many times the scan has to start a new flood |
| Pacific Atlantic Water Flow | `graphs/pacific_atlantic_water_flow.py` | climb uphill from each ocean's border; answer = cells both floods reach |
| Rotting Oranges | `graphs/rotting_oranges.py` · `practice/simple/44_rotting_oranges.py` | multi-source BFS from all rotten oranges; one ring = one minute; -1 if fresh ones remain |
| Shortest Path in a Grid with Obstacles Elimination | `graphs/shortest_path_in_a_grid_with_obstacles_elimination.py` | state = (row, col, eliminations left); more budget at the same cell is a different state |
| Shortest Path to Get All Keys | `graphs/shortest_path_to_get_all_keys.py` | state = (row, col, key bitmask); the goal is any cell with the full mask |
| Shortest Path Visiting All Nodes | `graphs/shortest_path_visiting_all_nodes.py` | state = (node, visited mask); seed all n starts; the first full mask wins |
| Sliding Puzzle | `graphs/sliding_puzzle.py` | the board as a 6-char string; neighbours = swap the blank with a touching index |
| Surrounded Regions | `graphs/surrounded_regions.py` | mark the O's reachable from the border as safe; flip every other O |
| Walls and Gates | `graphs/walls_and_gates.py` | multi-source BFS from all gates; the first write into a room is final, so INF itself means "unvisited" |
| Word Ladder | `graphs/word_ladder.py` | BFS on words; wildcard buckets ("h*t") list neighbours without comparing all pairs |
| Word Ladder II | `graphs/word_ladder_ii.py` | BFS by rings keeping every parent from the previous ring; backtrack from endWord |

### Self-check

1. Why mark a node as seen when you push it, not when you pop it?
<details><summary>Answer</summary>Between being pushed and being popped, a node can be discovered again by other neighbours. If it is only marked at pop time, every discovery pushes another copy: the queue grows, work repeats, and anything counted per pop (like an area) is counted more than once. Marking on push lets each node into the container exactly once. (Marking on pop also works if you skip nodes that are already marked; that is the form Dijkstra needs.)</details>

2. Why does BFS give the fewest moves while DFS does not?
<details><summary>Answer</summary>The queue only ever holds distances <code>d</code> and <code>d + 1</code>, in that order, so nodes come out in order of distance; when the target first comes out, no shorter path can still be waiting. DFS follows whichever corridor it tried first, so it reaches the target along <em>some</em> path, not necessarily a short one.</details>

3. In Shortest Path to Get All Keys, why can't <code>seen</code> hold just the cell?
<details><summary>Answer</summary>The same cell with more keys is a different situation: locks that were walls are now open. A walk often has to come back over cells it has already crossed, for example to fetch a key behind it. With cell-only <code>seen</code>, <code>["a.@A.b"]</code> answers -1 instead of 7.</details>

4. Why should the move count never be part of the node?
<details><summary>Answer</summary>The queue order already tells you the distance (or the distance rides along beside the state). With the count inside, the same cell reached after 3 moves and after 5 moves are two different nodes, so <code>seen</code> cannot merge them and the search re-explores places it has already been: on an open 6 × 6 grid, 111 pops instead of 36.</details>

<a id="s18"></a>

## Graphs II: Ordering & Connectivity

> Two questions about how nodes relate. **Order**: *what can I do first?* Keep taking a node that nothing is waiting on (topological sort). **Connectivity**: *are these two in the same group?* Every group has a boss; merging two groups means pointing one boss at the other (union-find).

[Graphs I](#s17) walked a graph to measure distances and regions. This section asks two other questions, in what order the nodes can be taken and which nodes belong together, and answers each with a small structure instead of a fresh search.

**Reach for it when** the problem talks about order: prerequisites, dependencies, "must come before", a build order, an alphabet hidden in sorted words, or the minimum time when tasks run in parallel. Reach for it too when it talks about groups: merge, same group, connected, edges that arrive one at a time, "the extra edge that makes a cycle", "after each operation, how many groups?", or queries with a weight limit.

"Which cable, if cut, disconnects the network?" is bridges, and "use every ticket exactly once" is an Eulerian path; both come in the second pass at the end of Variations.

**In this repo:** `graphs/` (19 of its 45 problems) · bank: `practice/simple/43_course_schedule.py`, `practice/simple/47_redundant_connection.py` · basics: `practice/simple/basics/graphs/02_topological_sort_kahn_and_dfs.py`, `practice/simple/basics/graphs/03_union_find.py`

### The picture

```text
TOPOLOGICAL SORT (Kahn): keep taking whatever nothing is waiting on

      0 -----> 1            indegree = how many arrows point INTO a node
      |        |
      v        v            start      indegree  0:0  1:1  2:1  3:2    ready [0]
      2 -----> 3            take 0  ->           1:0  2:0  3:2         ready [1, 2]
                            take 1  ->                     3:1         ready [2]
                            take 2  ->                     3:0         ready [3]
                            take 3  ->  order 0 1 2 3

A cycle 1 -> 2 -> 3 -> 1: each waits on another, none ever reaches 0, and "ready" runs dry early.
```

```text
UNION-FIND: every group is a tree, and its root is the group's name

  union(0, 1)   union(2, 3)          union(1, 3): find(1) = 0, find(3) = 2,
                                     so hang root 2 under root 0
     0     2                              0
     |     |                             / \
     1     3                            1   2
                                            |
                                            3      find(3) walks 3 -> 2 -> 0; path halving then
                                                   points 3 at its grandparent, the root
  union(0, 2) now: find(0) == find(2) == 0, so it returns False: they were already connected.
```

Kahn's algorithm gives an order and a cycle check in one pass: it is iterative, so it never hits the recursion limit, and a cycle shows up as `len(order) < n`. DFS colours are the choice when you are already inside a DFS, or when the question is which nodes can reach a cycle: Find Eventual Safe States (802) asks for the nodes from which every path ends at a node with no way out, which are exactly the nodes that cannot reach a cycle.

For connectivity, a graph given once needs nothing more than one BFS or DFS ([Graphs I](#s17)). Union-find earns its place when edges arrive over time, when many "connected?" questions come between the arrivals, or when the question is the edge that closes a cycle.

**Why it is fast.** The brute-force ordering rescans every edge each round to find a node that is ready: O(V · (V + E)). Kahn keeps one counter per node, and taking a node only changes the counters of *its* successors, so each edge is touched once: O(V + E).

For connectivity, re-running a BFS for every "connected?" question costs O(V + E) each time. Groups only ever merge, so a forest of parent pointers can answer in nearly O(1), as long as the trees stay flat.

Union by size keeps them flat: a node goes one level deeper only when its group is hung under a group at least as big, so its group at least doubles each time, and no tree grows deeper than log₂ n. Path halving flattens the rest. Together they cost α(n) per operation amortised, that is, averaged over any long run of operations; α is the inverse Ackermann function, which stays below 5 for any n you will ever store.

### From idea to code

**The idea in two sentences:** *Order: count each node's unfinished predecessors, take any node whose count is 0, and taking it lowers its successors' counts.* *Connectivity: `find` walks up to the root that names a group, `union` points one root at the other, and a `union` whose two roots are already equal has just found an edge that closes a cycle.*

For Kahn's algorithm the **state** is three things: the graph, where `graph[u]` lists every v that u must precede, an `indegree` list, and a queue of ready nodes. The **definition**: `indegree[v]` counts the predecessors of v not taken yet, where a node's indegree is the number of arrows pointing into it. The **invariant**: every node in the queue has all its predecessors already in `order`. A **step** pops u, appends it to `order`, and lowers `indegree[v]` for each successor v, pushing v the moment its count reaches 0.

The **record** is the pop order itself, which is a valid order. **Init** puts *every* node of indegree 0 in the queue, isolated nodes too. The **return** is `order` when `len(order) == n`; anything shorter means a cycle, translated into `[]`, `False` or `""`.

Union-find makes the same decisions about groups. Its state is `parent[x]`, `size[root]` and, when asked, the number of groups. By definition `find(x)` is the root of x's tree, and two nodes share a group exactly when they share a root. The invariant: every group is one tree, and `r` is a root exactly when `parent[r] == r`. A step, `union(a, b)`, finds both roots and, if they differ, hangs the smaller tree under the bigger.

`union` also records: it returns `False` when the two were already connected, so the edge closes a cycle, and `True` when there is one group fewer. Init is `parent = list(range(n))`, `size = [1] * n` and `count = n`, and the return is the count, the redundant edge, or the groups bucketed by `find`.

Course Schedule asks whether n courses can all be taken when a pair `[a, b]` means "take b before a"; Course Schedule II asks for one such order, or `[]` when none exists. With 4 courses and `[[1, 0], [2, 0], [3, 1], [3, 2]]`, one order is `[0, 1, 2, 3]`. The template keeps one convention, an edge (u, v) for "u before v", so the wrapper converts the pairs once and never thinks about them again. In the code, "everything I can do right now" is the starting queue, and "u is done, so v waits on one fewer" is `indegree[v] -= 1`.

In [ ]:
def topo_kahn(n, edges):
    """Edges (u, v) mean "u before v". Returns an order of 0..n-1, or [] if there is a cycle."""
    graph = [[] for _ in range(n)]
    indegree = [0] * n                            # STATE: indegree[v] = predecessors of v not taken yet
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(v for v in range(n) if indegree[v] == 0)   # STATE + INIT: everything ready right now
    order = []
    while queue:
        u = queue.popleft()                       # any ready node will do
        order.append(u)                           # RECORD: the pop order is a valid order
        for v in graph[u]:
            indegree[v] -= 1                      # STEP: u is done, so v waits on one fewer
            if indegree[v] == 0:                  # u was v's last missing predecessor
                queue.append(v)
    return order if len(order) == n else []       # RETURN: anyone never freed sits on or behind a cycle


def find_order(n, prerequisites):                 # Course Schedule II: [a, b] means "take b before a"
    return topo_kahn(n, [(b, a) for a, b in prerequisites])   # convert once, then think u -> v


print(find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))   # [0, 1, 2, 3]
print(find_order(3, [[0, 1], [1, 2], [2, 0]]))           # []  (0 needs 1, 1 needs 2, 2 needs 0)
print(find_order(3, []))                                 # [0, 1, 2]  (no rules: any order)

**Try it**
- Convert the wrong way in `find_order` (`(a, b)` instead of `(b, a)`): the first example gives `[3, 1, 2, 0]`, the order backwards. The second still gives `[]`, because reversing every edge keeps every cycle. That is how this bug survives Course Schedule I and only shows up in II.
- The smallest order: replace the deque with a heap (`heapq.heapify`, `heappop`, `heappush`) and run `topo_kahn(6, [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)])`: `[4, 5, 0, 2, 3, 1]` instead of the deque's `[4, 5, 2, 0, 3, 1]`.
- Replace `popleft()` with `pop()` (a stack): the first example gives `[0, 2, 1, 3]`, a different order that is just as valid. Kahn does not care which ready node goes first.
- Predict `find_order(4, [[1, 0], [2, 1], [3, 2], [1, 3]])` before running it: `[]`. Course 0 is taken, but 1, 2 and 3 wait on each other in a circle.

The DFS version returns the same thing, an order or `[]`, and it needs three colours, not two. A node reached again can be *finished*, reached earlier along another path, which is harmless, or still *on the current path*, which means a cycle; "on the path I am walking right now" is `color[v] == GRAY`. A node finishes after everything it leads to, so the reversed finishing order is a topological order: the six-node graph below gives `[5, 4, 2, 3, 1, 0]`.

In [ ]:
WHITE, GRAY, BLACK = 0, 1, 2                      # not visited, on the current path, finished


def topo_dfs(n, edges):
    """Edges (u, v) mean u before v. Returns reversed DFS finishing order, or [] on a cycle."""
    graph = [[] for _ in range(n)]
    for u, v in edges:
        graph[u].append(v)
    color, finished = [WHITE] * n, []             # STATE + INIT: colours; nodes in the order they finish

    def visit(u):                                 # False means "found a cycle"
        color[u] = GRAY                           # u is on the path we are walking right now
        for v in graph[u]:
            if color[v] == GRAY:                  # stepped back onto our own path: a cycle
                return False
            if color[v] == WHITE and not visit(v):
                return False
        color[u] = BLACK                          # everything reachable from u is finished
        finished.append(u)                        # STEP: so u finishes after all of its successors
        return True

    for u in range(n):
        if color[u] == WHITE and not visit(u):
            return []
    return finished[::-1]                         # RETURN: reversed finishing order


print(topo_dfs(6, [(5, 2), (5, 0), (4, 0), (4, 1), (2, 3), (3, 1)]))   # [5, 4, 2, 3, 1, 0]
print(topo_dfs(3, [(0, 1), (0, 2), (2, 1)]))                           # [0, 2, 1]  (1 is reached twice: no cycle)
print(topo_dfs(3, [(0, 1), (1, 2), (2, 0)]))                           # []

**Try it**
- Use two colours only: change `if color[v] == GRAY` to `if color[v] != WHITE`. The diamond `[(0, 1), (0, 2), (2, 1)]` now comes back `[]`: node 1 was finished, not on the path, but the two look the same.
- Print `finished` before reversing: `[0, 1, 3, 2, 4, 5]` for the first example. The deepest nodes finish first.
- Return `finished` without reversing: every edge now points backwards. It is a valid order for the graph with all arrows flipped.

Union-find answers "are a and b in the same group?" while groups keep merging. After `union(0, 1)`, `union(2, 3)` and `union(1, 3)`, nodes 0 and 3 share a root, so `union(0, 2)` returns False: they were already together. "Merge the two groups" is `parent[find(b)] = find(a)`, which links roots and never the nodes themselves. Path halving makes each node on the way up point at its grandparent, and union by size hangs the smaller tree below the bigger one.

In [ ]:
class DSU:
    def __init__(self, n):
        self.parent = list(range(n))              # STATE + INIT: everyone starts as their own root
        self.size = [1] * n                       # STATE + INIT: size[root] = members of that group
        self.count = n                            # STATE + INIT: number of groups

    def find(self, x):                            # the root that names x's group
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path halving: skip to the grandparent
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                          # RECORD: already connected (this edge closes a cycle)
        if self.size[ra] < self.size[rb]:
            ra, rb = rb, ra                       # union by size: the smaller tree hangs below
        self.parent[rb] = ra                      # STEP: two groups become one
        self.size[ra] += self.size[rb]
        self.count -= 1
        return True


dsu = DSU(6)
print([dsu.union(a, b) for a, b in [(0, 1), (2, 3), (1, 3), (0, 2)]])   # [True, True, True, False]
print(dsu.count, dsu.find(3) == dsu.find(0), dsu.size[dsu.find(0)])      # 3 True 4

**Try it**
- On a fresh `DSU(6)` do the unions `(0, 1)`, `(2, 3)`, `(1, 3)`, print `parent`, call `find(3)`, and print again: `[0, 0, 0, 2, 4, 5]` becomes `[0, 0, 0, 0, 4, 5]`. Node 3 now points at its old grandparent, the root.
- Link nodes instead of roots: change `self.parent[rb] = ra` to `self.parent[b] = a`. Then `d = DSU(3); d.union(0, 1); d.union(2, 1); print(d.find(0) == d.find(1))` prints False: node 1 was pulled out from under 0.
- Print `dsu.size`: `[4, 1, 2, 1, 1, 1]`. Only a root's entry means something; `size[2] = 2` is left over from when 2 was a root.
- Delete the two `if ra == rb` lines and rerun the cell: the last union returns True and `count` drops to 2, although the groups are still `{0, 1, 2, 3}`, `{4}` and `{5}`.

In an interview, two functions are enough. Path halving alone keeps `find` fast in practice; add union by size if you are asked for the guarantee:

```py
parent = list(range(n))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]      # path halving: skip to the grandparent
        x = parent[x]
    return x

def union(a, b):                           # False: a and b were already connected
    ra, rb = find(a), find(b)
    if ra == rb:
        return False
    parent[ra] = rb
    return True
```

### Watch it work

Two traces make the templates visible. `trace_kahn` runs Kahn's algorithm on the picture's graph and prints the indegrees and the ready queue after each take. `trace_redundant` solves Redundant Connection, which gives a tree on nodes 1..n plus one extra edge and asks which edge to remove, the last one in the input if several work: union the edges in order, and the first union that finds both ends already connected names the edge, `[1, 4]` here.

In [ ]:
def trace_kahn(n, edges):                         # edges (u, v): u before v
    graph, indegree = [[] for _ in range(n)], [0] * n
    for u, v in edges:
        graph[u].append(v)
        indegree[v] += 1
    queue = deque(v for v in range(n) if indegree[v] == 0)
    print(f"start:  indegree={indegree}  queue={list(queue)}")
    while queue:
        u = queue.popleft()
        for v in graph[u]:
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
        print(f"take {u}: indegree={indegree}  queue={list(queue)}")


def trace_redundant(edges):                       # nodes are 1..n
    dsu = DSU(len(edges) + 1)
    for u, v in edges:
        merged = dsu.union(u, v)
        print(f"edge {u}-{v}: {'merge' if merged else 'already connected -> redundant'}  parent[1:]={dsu.parent[1:]}")
        if not merged:
            return [u, v]


trace_kahn(4, [(0, 1), (0, 2), (1, 3), (2, 3)])
print(trace_redundant([[1, 2], [2, 3], [3, 4], [1, 4], [1, 5]]))   # [1, 4]

**Try it**
- `trace_kahn(3, [(0, 1), (1, 2), (2, 1)])`: after taking 0 the queue is empty while 1 and 2 still wait on each other. That is what a cycle looks like from inside Kahn's algorithm.
- `trace_redundant([[1, 2], [1, 3], [2, 3]])` returns `[2, 3]`: the triangle closes on its last edge.
- Shuffle the input: `trace_redundant([[1, 4], [3, 4], [1, 3], [1, 2], [4, 5]])` returns `[1, 3]`. The answer is whichever edge of the cycle comes last.

### Where it goes wrong

1. **Edge direction.** `[a, b]` means "b before a", so the edge is `b -> a`. Reversed edges give the order backwards (`[3, 1, 2, 0]` above), and Course Schedule I still passes because reversing edges keeps cycles, so the bug hides until part II. Convert once, at the top.
2. **Forgotten nodes.** Seed the queue with *every* node of indegree 0, including nodes with no edges at all. Alien Dictionary, which recovers an alphabet from words sorted in it, makes every letter a node, even one that appears in no rule: `["ab", "adc"]` must output `c` too.
3. **Duplicate edges with a set adjacency.** If `graph[x]` is a set, add to `indegree[y]` only when `y` was really new. Without that guard, `alien_order(["ab", "ac", "bb", "bc"])` returns `""` instead of `"abc"`: the fact "b before c" was counted twice but can only be freed once. With list adjacency duplicates are harmless: counted twice, freed twice.
4. **Two colours in DFS.** "Visited" is not "on my path": with two colours, the diamond `0->1, 0->2, 2->1` looks like a cycle.
5. **Linking nodes instead of roots.** `parent[b] = a` can pull `b` out of its old group (`union(0, 1)`, then `union(2, 1)` leaves 0 and 1 apart). Always link `find(b)` under `find(a)`.
6. **No halving, no union by size.** With a plain `parent[find(a)] = find(b)`, the unions `union(0, k)` for k = 1 … 999 build one long chain, and `find(0)` then walks 999 links.
7. **Counting every union.** Lower the group count only when the roots differed; a repeated edge must not change it. Four nodes with the edges `[0, 1]`, `[1, 0]` and `[2, 3]` form 2 groups, but counting every edge, `n - len(edges)`, says 1.
8. **1-indexed nodes.** Courses, people and tree nodes are often numbered `1..n`: `DSU(3).union(1, 3)` raises `IndexError`. Allocate `n + 1`.
9. **Comparing parents instead of roots.** After `union(0, 1)`, `union(2, 3)`, `union(0, 2)` the parents are `[0, 0, 0, 2]`: `parent[3] == parent[1]` is False although `find(3) == find(1)`. Only roots name groups.

### Edge cases to say out loud

One node · no edges (any order is valid, every node is its own group) · a self-loop (a cycle of length 1; `union(x, x)` is False) · duplicate edges · several separate chains · a word that comes before its own prefix (Alien Dictionary) · one node and no edges is a valid tree.

In [ ]:
def respects(order, edges):                       # does every edge (u, v) have u earlier than v?
    pos = {x: i for i, x in enumerate(order)}
    return all(pos[u] < pos[v] for u, v in edges)


assert find_order(1, []) == [0]                                     # one course, nothing to wait for
assert find_order(2, [[1, 1]]) == []                                # a self-loop is a cycle of length 1
assert find_order(3, [[1, 0], [1, 0]]) == [0, 2, 1]                 # a duplicate edge is counted AND freed twice
assert respects(topo_dfs(4, [(0, 1), (2, 3)]), [(0, 1), (2, 3)])    # two separate chains
assert respects(topo_dfs(5, []), [])                                # no edges: any order is valid
d = DSU(3)
assert d.union(0, 0) is False and d.count == 3                      # a node is already with itself
assert d.union(0, 1) and d.union(1, 2) and d.count == 1
print("edge cases pass")

**Try it**
- Predict `find_order(2, [[0, 1], [1, 0]])` before running it (`[]`: two courses waiting on each other).
- `respects([0, 1, 2], [(2, 0)])` is False. Test any order you produce with `respects` instead of comparing it with one "expected" order: many orders are correct.
- `topo_dfs(1, [(0, 0)])` returns `[]`: node 0 is still GRAY when it looks at itself.

### Variations

Every variation keeps one of the two templates and changes what goes in or what is read out: which nodes start, what an edge means, or what is done with the groups at the end.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Is any order possible?** | count the pops; possible iff all n come out | Course Schedule (207) |
| **Return an order** | keep the pop order | Course Schedule II (210) |
| **Smallest or unique order** | a heap instead of the deque; the order is unique iff the queue never holds two nodes | Sequence Reconstruction (444): is one given sequence the only order the rules allow |
| **Derive the edges first** | compare neighbouring words; their first different letter is one edge; prefix check | Alien Dictionary (269) |
| **Longest path in a DAG** | in topological order, push `finish[v] = max(finish[v], finish[u] + time[v])` | Parallel Courses III (2050) |
| **Peel leaves** | Kahn on an undirected tree: remove degree-1 leaves ring by ring; the last 1–2 nodes are the centre | Minimum Height Trees (310) |
| **Tree check / component count / judge** | n − 1 edges and no failed union; `dsu.count`; in-degree minus out-degree | Graph Valid Tree (261), Number of Connected Components (323), Find the Town Judge (997) |
| **The cycle edge, undirected** | the first union that returns False | Redundant Connection (684) |
| **Group by a shared key** | each key (an email) remembers its first owner; union with it | Accounts Merge (721) |
| **Union, then act per component** | sort letters inside each component; stones − groups | Smallest String With Swaps (1202), Most Stones Removed (947) |
| **Equations as unions** | union every `a == b`, then check that no `a != b` falls inside one group | Satisfiability of Equality Equations (990): can every `==` and `!=` hold at once |
| **Two-colouring with a DSU** | every neighbour of u joins one group, and that group must not contain u | Is Graph Bipartite (785): two camps, every edge between them, as in [Graphs I](#s17) |
| **Online counting** | a new node adds 1; each successful union subtracts 1 | Number of Islands II (305) |
| *Second pass:* **bridges** | DFS low-link: tree edge (u, v) is a bridge iff `low[v] > disc[u]` | Critical Connections in a Network (1192) |
| *Second pass:* **use every edge once** | Hierholzer: write a node down when it is stuck, reverse at the end | Reconstruct Itinerary (332) |
| *Second pass:* **prime hubs** | each prime factor remembers its first owner; union with it | Largest Component Size by Common Factor (952) |
| *Second pass:* **unions over time** | one timestamp at a time; reset everyone not connected to person 0 | Find All People With Secret (2092) |
| *Second pass:* **offline thresholds** | sort queries by limit, let edges in by weight, answer into the original slots | Checking Existence of Edge Length Limited Paths (1697) |
| *Second pass:* **an order inside an order** | topo-sort the items and the groups separately; output groups in order, items inside | Sort Items by Groups Respecting Dependencies (1203) |
| *Second pass:* **the cycle edge, directed** | a node with two parents first: try dropping the later edge, else the earlier one | Redundant Connection II (685) |
| *Second pass:* **two players** | two DSUs; shared edges first, then each player's own | Remove Max Number of Edges to Keep Graph Fully Traversable (1579) |
| *Second pass:* **facts per component** | per root: its size and how many infected nodes it holds | Minimize Malware Spread (924) |

Three warm-ups need almost nothing beyond the templates. Graph Valid Tree asks whether n nodes and a list of undirected edges form one tree, and `5, [[0, 1], [0, 2], [0, 3], [1, 4]]` does: a tree on n nodes has exactly n − 1 edges, none of them wasted on a loop. Number of Connected Components counts the groups, 2 for `5, [[0, 1], [1, 2], [3, 4]]`: start at n, and every real merge removes one.

Find the Town Judge looks for the one person who is trusted by everybody else and trusts nobody, or -1: with 3 people and `[[1, 3], [2, 3]]`, the judge is 3. It needs no graph at all, only a score per person, in-degree minus out-degree, and only the judge reaches n − 1.

In [ ]:
def valid_tree(n, edges):                         # exactly n - 1 edges, and none of them closes a cycle
    dsu = DSU(n)
    return len(edges) == n - 1 and all(dsu.union(u, v) for u, v in edges)


def count_components(n, edges):                   # start with n groups; every real merge removes one
    dsu = DSU(n)
    for u, v in edges:
        dsu.union(u, v)
    return dsu.count


def find_judge(n, trust):                         # degrees only: trusted by n - 1 people, trusts nobody
    score = [0] * (n + 1)
    for a, b in trust:
        score[a] -= 1                             # trusting anyone disqualifies a
        score[b] += 1
    return next((i for i in range(1, n + 1) if score[i] == n - 1), -1)


print(valid_tree(5, [[0, 1], [0, 2], [0, 3], [1, 4]]), valid_tree(5, [[0, 1], [1, 2], [2, 3], [1, 3], [1, 4]]))   # True False
print(count_components(5, [[0, 1], [1, 2], [3, 4]]))                                                           # 2
print(find_judge(3, [[1, 3], [2, 3]]), find_judge(3, [[1, 3], [2, 3], [3, 1]]), find_judge(1, []))               # 3 -1 1

**Try it**
- Drop the `len(edges) == n - 1` check: `valid_tree(4, [[0, 1], [2, 3]])` becomes True. No cycle, but two pieces; the edge count is what rules that out.
- Delete `score[a] -= 1` in `find_judge`: `find_judge(3, [[1, 3], [2, 3], [3, 1]])` returns 3, a "judge" who trusts somebody.
- `count_components(4, [[0, 1], [1, 0], [2, 3]])` is 2: the repeated edge's union returns False, so the count does not drop.
- `valid_tree(1, [])` is True: a single node with no edges is a tree.

Sometimes the edges are hidden, and the first job is to find them. Alien Dictionary gives words sorted in an unknown alphabet and asks for that alphabet, or `""` when no alphabet fits: `["wrt", "wrf", "er", "ett", "rftt"]` gives `"wertf"`. Two *neighbouring* words give at most one fact, at their first different letter, where the earlier word's letter comes first; the letters after that difference say nothing. And a word placed before its own prefix, `"abc"` before `"ab"`, is impossible in any alphabet.

In [ ]:
def alien_order(words):
    graph = {c: set() for w in words for c in w}  # every letter is a node, even one with no rules
    indegree = {c: 0 for c in graph}
    for w1, w2 in zip(words, words[1:]):          # only NEIGHBOURING words are compared
        for x, y in zip(w1, w2):
            if x != y:                            # the first difference is the only fact: x before y
                if y not in graph[x]:             # a set, so each edge is counted once
                    graph[x].add(y)
                    indegree[y] += 1
                break
        else:                                     # no difference in the common part...
            if len(w1) > len(w2):                 # ...and "abc" sits before "ab": impossible
                return ""
    queue = deque(c for c in graph if indegree[c] == 0)
    order = []
    while queue:
        c = queue.popleft()
        order.append(c)
        for nxt in graph[c]:
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)
    return "".join(order) if len(order) == len(graph) else ""


print([alien_order(ws) for ws in (["wrt", "wrf", "er", "ett", "rftt"], ["z", "x", "z"], ["abc", "ab"])])   # ['wertf', '', '']

**Try it**
- Delete the `else:` branch (the prefix check): `alien_order(["abc", "ab"])` returns `"abc"` instead of `""`.
- Delete the `break`: `alien_order(["ab", "ba"])` returns `""` instead of `"ab"`, because the letters after the first difference produced the false fact "b before a".
- `alien_order(["ab", "adc"])` is `"abcd"`: `c` appears in no rule but still belongs to the alphabet.
- `alien_order(["z", "z"])` is `"z"`: equal words give no rule and no contradiction.

A topological order can also carry values forward, which turns Kahn's algorithm into a longest-path algorithm. Parallel Courses III gives each course a duration, lets any number of courses run at once, and asks for the fewest months to finish them all: with `time = [3, 2, 5]` and course 3 waiting for courses 1 and 2, the answer is 8. A course starts when its *slowest* prerequisite ends, so `finish[v] = time[v] + max(finish of its prerequisites)`, and in Kahn's order every prerequisite has reported before v is popped.

In [ ]:
def minimum_time(n, relations, time):
    graph, indegree = [[] for _ in range(n + 1)], [0] * (n + 1)
    for a, b in relations:                        # courses are 1..n; a must finish before b starts
        graph[a].append(b)
        indegree[b] += 1
    finish = [0] + time                           # finish[c] = earliest month c can end (so far)
    queue = deque(c for c in range(1, n + 1) if indegree[c] == 0)
    while queue:
        u = queue.popleft()                       # every prerequisite of u has reported: finish[u] is final
        for v in graph[u]:
            finish[v] = max(finish[v], finish[u] + time[v - 1])   # v cannot end before u ends + v's own time
            indegree[v] -= 1
            if indegree[v] == 0:
                queue.append(v)
    return max(finish)


print(minimum_time(3, [[1, 3], [2, 3]], [3, 2, 5]))                                 # 8
print(minimum_time(5, [[1, 5], [2, 5], [3, 5], [3, 4], [4, 5]], [1, 2, 3, 4, 5]))   # 12

**Try it**
- Add `time[u - 1]` again (`finish[u] + time[u - 1] + time[v - 1]`): the first example says 11. `finish[u]` already includes u's own time.
- Print `finish` at the end: `[0, 3, 2, 8]`. Course 3 starts at month 3, when its slower prerequisite ends.
- With no relations, `minimum_time(3, [], [3, 2, 5])` is 5: everything runs in parallel.
- Chain them: `minimum_time(3, [[1, 2], [2, 3]], [3, 2, 5])` is 10.

Kahn's idea also works on an undirected tree, with "degree 1" in place of "indegree 0". Minimum Height Trees asks for every root that gives a tree its smallest height: the star with centre 1 has the single answer `[1]`. The best root is the centre, and the centre is what is left after peeling the tree like an onion: remove every leaf at once, then the new leaves, until at most two nodes remain.

In [ ]:
def find_min_height_trees(n, edges):
    if n <= 2:
        return list(range(n))
    graph, degree = [[] for _ in range(n)], [0] * n
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
        degree[u] += 1
        degree[v] += 1
    leaves = deque(v for v in range(n) if degree[v] == 1)   # like Kahn's indegree-0 queue
    remaining = n
    while remaining > 2:
        for _ in range(len(leaves)):              # peel one whole ring of leaves at once
            leaf = leaves.popleft()
            remaining -= 1
            for nb in graph[leaf]:
                degree[nb] -= 1
                if degree[nb] == 1:               # nb just became a leaf of what is left
                    leaves.append(nb)
    return sorted(leaves)


print(find_min_height_trees(4, [[1, 0], [1, 2], [1, 3]]))                   # [1]
print(find_min_height_trees(6, [[3, 0], [3, 1], [3, 2], [3, 4], [5, 4]]))   # [3, 4]
print(find_min_height_trees(1, []))                                         # [0]

**Try it**
- Peel one leaf at a time (drop the `for _ in range(len(leaves))` line and keep its body): the star in the first example answers `[1, 3]` instead of `[1]`. A ring of leaves must go together.
- Change `while remaining > 2` to `> 1`: the second example answers `[]` instead of `[3, 4]`. A tree can have two centres.
- Predict the centre of the path 0-1-2-3-4, then check: `find_min_height_trees(5, [[0, 1], [1, 2], [2, 3], [3, 4]])` is `[2]`; with only four nodes in a line it is `[1, 2]`.

Union-find takes over when groups come from shared keys. Accounts Merge gives accounts as a name followed by emails and merges any two that share an email, since one person owns both; a shared name proves nothing. Below, the two Johns who share `b@m` merge, and the John with only `x@m` stays apart. Instead of comparing every pair of accounts, O(n²), each email remembers the *first* account that listed it, every later account with that email is unioned with it, and the emails are finally bucketed by their group's root.

In [ ]:
def accounts_merge(accounts):
    dsu = DSU(len(accounts))
    owner = {}                                    # email -> the first account that listed it
    for i, (_, *emails) in enumerate(accounts):
        for e in emails:
            if e in owner:
                dsu.union(i, owner[e])            # a shared email: same person
            else:
                owner[e] = i
    groups = defaultdict(list)
    for e, i in owner.items():
        groups[dsu.find(i)].append(e)             # bucket every email under its group's root
    return sorted([accounts[root][0]] + sorted(es) for root, es in groups.items())


print(accounts_merge([["John", "a@m", "b@m"], ["John", "b@m", "c@m"], ["Mary", "d@m"], ["John", "x@m"]]))
# [['John', 'a@m', 'b@m', 'c@m'], ['John', 'x@m'], ['Mary', 'd@m']]

**Try it**
- Print `owner` and `dsu.parent` after the first loop: `b@m` belongs to account 0, and account 0 now hangs under account 1. The two Johns who share nothing stay apart, although they have the same name.
- Delete the `else:` branch (never record an owner): the result is `[]`. `owner` is both the shared-email detector and the list of emails to print.
- Run `accounts_merge([["A", "x"], ["A", "y"], ["A", "x", "y"]])`: `[['A', 'x', 'y']]`. The third account bridges two that share nothing.

Some problems only need the groups: union everything first, then do one thing per component. Smallest String With Swaps lets you swap the letters at any listed pair of indexes, as often as you like, and asks for the smallest string you can reach: `"dcab"` with the pairs `[0, 3]` and `[1, 2]` becomes `"bacd"`. Swaps chain, so the letters inside a component can be put in *any* order, and the answer sorts them.

Most Stones Removed lets you remove a stone that shares a row or a column with another stone still on the board, and asks for the most stones you can remove: 5 of the 6 stones below. A stone glues its row to its column, and every group of stones can be cleared down to one stone, so the answer is the number of stones minus the number of groups.

In [ ]:
def smallest_string_with_swaps(s, pairs):
    dsu = DSU(len(s))
    for a, b in pairs:
        dsu.union(a, b)                     # swaps chain: any order inside a component is reachable
    groups = defaultdict(list)
    for i in range(len(s)):
        groups[dsu.find(i)].append(i)       # one component's indexes, increasing
    out = list(s)
    for idx in groups.values():
        for i, ch in zip(idx, sorted(s[i] for i in idx)):
            out[i] = ch                     # smallest letters to the smallest indexes
    return "".join(out)


def remove_stones(stones):
    dsu = DSU(20002)                        # rows 0..10000 and columns 10001..20001 share one DSU
    for r, c in stones:
        dsu.union(r, 10001 + c)             # a stone glues its row to its column
    groups = {dsu.find(r) for r, _ in stones}
    return len(stones) - len(groups)        # each group can be cleared down to one stone


print(smallest_string_with_swaps("dcab", [[0, 3], [1, 2]]), smallest_string_with_swaps("dcab", [[0, 3], [1, 2], [0, 2]]),
      smallest_string_with_swaps("cba", [[0, 1], [1, 2]]))                                     # bacd abcd abc
print(remove_stones([[0, 0], [0, 1], [1, 0], [1, 2], [2, 1], [2, 2]]), remove_stones([[0, 0], [0, 2], [1, 1], [2, 0], [2, 2]]),
      remove_stones([[0, 0]]))                                                                # 5 3 0

**Try it**
- Print `dict(groups)` for `"dcab", [[0, 3], [1, 2]]`: `{0: [0, 3], 1: [1, 2]}`. Index 0 can only trade with index 3, so sorting the whole string (`"abcd"`) would be wrong; the answer is `"bacd"`.
- Forget the column offset (`dsu.union(r, c)`): `remove_stones([[0, 1], [1, 2]])` answers 1 instead of 0, because column 1 got glued to row 1.
- `remove_stones([[0, 0], [5, 5]])` is 0: two stones that share neither a row nor a column.

The last union-find variation counts groups while they form. Number of Islands II turns water into land one cell at a time and asks for the island count after every step: on a 3 × 3 grid, adding (0, 0), (0, 1), (1, 2) and (2, 1) gives `[1, 1, 2, 3]`. Recounting the grid each time repeats almost all the work. Instead, a new cell is a new island, +1, and it swallows each *different* neighbouring island, −1 per successful union.

In [ ]:
def num_islands_2(m, n, positions):
    parent, count, out = {}, 0, []                # only land cells live in the union-find

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for r, c in positions:
        if (r, c) not in parent:                  # adding the same cell twice changes nothing
            parent[(r, c)] = (r, c)
            count += 1                            # a new island of one cell...
            for nb in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if nb in parent:                  # (only land cells are keys, so no bounds check)
                    ra, rb = find((r, c)), find(nb)
                    if ra != rb:
                        parent[ra] = rb
                        count -= 1                # ...that swallows each DIFFERENT neighbouring island
        out.append(count)
    return out


print(num_islands_2(3, 3, [[0, 0], [0, 1], [1, 2], [2, 1]]))           # [1, 1, 2, 3]
print(num_islands_2(3, 3, [[0, 0], [0, 2], [1, 1], [0, 1], [0, 1]]))   # [1, 2, 3, 1, 1]

**Try it**
- Remove the duplicate guard (make the `if` always true): `num_islands_2(3, 3, [[0, 0], [0, 0]])` reports `[1, 2]`, a second island that does not exist.
- Decrement for every land neighbour (drop the `if ra != rb`): `num_islands_2(2, 2, [[0, 0], [0, 1], [1, 0], [1, 1]])` gives `[1, 1, 1, 0]`. The last cell's two neighbours were already one island.
- Print `{cell: find(cell) for cell in parent}` at the end of the second example: all four cells report the same root.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Critical Connections in a Network gives a connected network of servers and asks for every cable whose removal disconnects some pair: in a triangle 0-1-2 with server 3 hanging off 1, only `[1, 3]` is critical. Removing each edge and re-running a BFS costs O(E · (V + E)); one DFS does it all.

Picture the DFS tree with every non-tree edge as a *rope* from a deep node up to an ancestor: cut the tree edge above v, and v's subtree stays attached exactly when some rope from inside it reaches above the cut.

```text
        0 <------+      tree edges go down; the non-tree edge 2-0 is a rope from 2 up to 0
        |        |
        1        |      cut 0-1 or 1-2: the rope from 2 still holds that piece, so neither is a bridge
       / \       |
      2   3      |      cut 1-3: nothing below 3 climbs above 1, so 3 falls off: a bridge
      |          |
      +----------+      disc = visiting order; low[v] = the smallest disc a rope from v's subtree reaches
```

In the code, `disc[v]` is the order in which the DFS visits v, and `low[v]` is the smallest `disc` a rope from v's subtree reaches. The tree edge (u, v) is a bridge exactly when `low[v] > disc[u]`: nothing below v climbs back to u or above it.

In [ ]:
def critical_connections(n, connections):
    graph = [[] for _ in range(n)]
    for u, v in connections:
        graph[u].append(v)
        graph[v].append(u)
    disc = [-1] * n                               # discovery time; -1 = not visited yet
    low = [0] * n                                 # smallest disc reachable from u's subtree with one back edge
    bridges, timer = [], 0

    def dfs(u, parent):
        nonlocal timer
        disc[u] = low[u] = timer
        timer += 1
        for v in graph[u]:
            if v == parent:
                continue                          # do not count the edge we came in on
            if disc[v] == -1:                     # tree edge: explore the child first
                dfs(v, u)
                low[u] = min(low[u], low[v])
                if low[v] > disc[u]:              # v's subtree cannot get back to u or above it
                    bridges.append([u, v])
            else:                                 # visited: an ancestor (back edge) or a finished descendant (larger disc: harmless)
                low[u] = min(low[u], disc[v])

    dfs(0, -1)
    return bridges


print(critical_connections(4, [[0, 1], [1, 2], [2, 0], [1, 3]]))           # [[1, 3]]
print(critical_connections(5, [[0, 1], [1, 2], [2, 0], [1, 3], [3, 4]]))   # [[3, 4], [1, 3]]

**Try it**
- Delete the `if v == parent: continue` lines: the first example returns `[]`. The edge back to the parent looks like a rope, so every subtree seems to have a way around.
- Change `low[v] > disc[u]` to `>=`: `[0, 1]` is reported too, although it sits on the triangle. `>=` belongs to the test for an articulation *node* (one whose removal disconnects; the root has its own rule), not to the bridge test.
- Print `disc` and `low` after `dfs(0, -1)`: `[0, 1, 2, 3]` and `[0, 0, 0, 3]`. Node 3's low (3) is bigger than node 1's disc (1): nothing below 3 reaches back, so 1-3 is a bridge.
- `critical_connections(5000, [[i, i + 1] for i in range(4999)])` raises `RecursionError`. Run `import sys; sys.setrecursionlimit(20_000)` first and it returns all 4999 edges. LeetCode allows n = 10⁵: raise the limit or make the DFS iterative.

Two more limits of this version. Parallel edges: `critical_connections(2, [[0, 1], [0, 1]])` gives `[[0, 1]]`, but removing one copy disconnects nothing; skip the parent *edge id*, not the parent node. A disconnected graph: `critical_connections(4, [[0, 1], [2, 3]])` misses the bridge 2-3, because the DFS has to be started from every unvisited node.

Reconstruct Itinerary gives plane tickets `[from, to]`, all used by one traveller who starts at JFK, and asks for the route that uses every ticket exactly once, the smallest in lexical order when several exist: the four tickets below fly JFK, MUC, LHR, SFO, SJC. Backtracking over ticket orders can dead-end deep down and undo a lot.

Hierholzer's algorithm never undoes a flight. Always fly the smallest unused ticket; when you are stuck at an airport, it must be the *end* of what is left, so write it down and back up one step. Airports are written in reverse finishing order, so the route is reversed at the end.

In [ ]:
def find_itinerary(tickets):
    graph = defaultdict(list)
    tickets = sorted(tickets, reverse=True)       # INIT: reverse-sorted, so pop() hands out the smallest
    for a, b in tickets:
        graph[a].append(b)
    route, stack = [], ["JFK"]
    while stack:
        while graph[stack[-1]]:                   # keep flying while this airport has unused tickets
            stack.append(graph[stack[-1]].pop())
        route.append(stack.pop())                 # stuck here: this airport is finished, write it down
    return route[::-1]                            # airports were written in reverse finishing order


print(find_itinerary([["MUC", "LHR"], ["JFK", "MUC"], ["SFO", "SJC"], ["LHR", "SFO"]]))   # ['JFK', 'MUC', 'LHR', 'SFO', 'SJC']
print(find_itinerary([["JFK", "KUL"], ["JFK", "NRT"], ["NRT", "JFK"]]))                  # ['JFK', 'NRT', 'JFK', 'KUL']

**Try it**
- Write each airport down on the way *in* instead (append it when you fly to it, no reverse): the second example becomes `['JFK', 'KUL', 'NRT', 'JFK']`, which needs a KUL → NRT ticket that does not exist.
- Sort ascending instead (drop `reverse=True`): `pop()` now hands out the *largest* destination. For `[["JFK", "SFO"], ["JFK", "ATL"], ["SFO", "ATL"], ["ATL", "JFK"], ["ATL", "SFO"]]` you get `['JFK', 'SFO', 'ATL', 'JFK', 'ATL', 'SFO']`: it uses every ticket, but it is not the smallest route.
- Print `route[-1], stack` each time an airport is written, for the second example: `KUL ['JFK']` comes first. KUL is a dead end, so it must end the trip; the loop through NRT is flown after it and written before the start, so the reversal puts KUL last.

Seven more Hard problems need no new code, only one more idea each:

- Largest Component Size by Common Factor (952) joins two numbers when they share a factor greater than 1 and asks for the size of the biggest group, 4 for `[4, 6, 15, 35]`. It is Accounts Merge with primes as the emails: union each number with the first owner of each of its prime factors, then count the numbers per root.
- Find All People With Secret (2092): person 0 tells `firstPerson` a secret at time 0, and a meeting `[x, y, t]` passes it between x and y, instantly along chains of meetings at the same time; who knows it in the end? Union-find can merge but never split, so union one timestamp's meetings at a time, then reset everyone from that timestamp who is not connected to person 0.
- Checking Existence of Edge Length Limited Paths (1697) asks, for each query `[p, q, limit]`, whether a path joins p and q using only edges lighter than the limit. Answer the queries offline: sort them by limit, let the edges in by weight, and each query becomes one `find` comparison, written into its original slot.
- Sort Items by Groups Respecting Dependencies (1203) asks for an order of items that respects every "before" rule and keeps each group's items together. Give every ungrouped item its own new group, topo-sort the items and, separately, the groups (`group[u] -> group[v]` when they differ), then output the groups in group order with each group's items in item order; either sort failing means `[]`.
- Redundant Connection II (685) adds one extra *directed* edge to a rooted tree and asks which edge to remove. If some node now has two parents, the answer is one of those two edges: skip the later one and union the rest; if no cycle appears, the later edge is the answer, and otherwise the earlier one is. If no node has two parents, return the first edge that closes a cycle.
- Remove Max Number of Edges to Keep Graph Fully Traversable (1579) asks how many edges can go while Alice and Bob, each with edges only they may use plus shared ones, still reach every node. Keep one DSU per player: shared edges go first and are kept when they merge anything, then each player's own. The answer is edges − kept, or −1 if either player's DSU still has more than one group.
- Minimize Malware Spread (924) asks which one node to take off the initially infected list so that the fewest nodes end up infected. Union the adjacency matrix, count the infected nodes per root, and remove the infected node whose component holds exactly one infected node and is the largest; ties go to the smallest index, and with no such node the answer is `min(initial)`.

### Say it in the interview

> "Prerequisites form a directed graph, and I need an order that respects every edge: a topological sort. The brute force rescans all edges for a course with nothing pending, O(V · (V + E)). I'll keep an indegree counter per course and a queue of courses at zero; taking a course only decrements its successors, so every edge is touched once, O(V + E). If fewer than n courses come out, the rest are stuck on a cycle."

> "Groups only ever merge, so I'll use union-find with path halving and union by size: amortised α(n) per operation, effectively constant. A union that finds both ends already under the same root is exactly the edge that closes a cycle."

Point at the `indegree[v] == 0` line, the `len(order) == n` check, and the `if ra == rb: return False` line: those three lines are the whole idea. Follow-ups to expect: *the smallest order?* (a heap instead of the deque), *is the order unique?* (no, if the queue ever holds two nodes at once), *show me the cycle* (DFS colours: when you meet a GRAY node, the GRAY path from it to the current node is the cycle), and *how fast is union-find?* (α(n) amortised, as above).

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Accounts Merge | `graphs/accounts_merge.py` | email → first account index; union on every repeat; bucket emails by root |
| Alien Dictionary | `graphs/alien_dictionary.py` | neighbouring words give one edge at their first difference; a word before its prefix → ""; Kahn |
| Checking Existence of Edge Length Limited Paths | `graphs/checking_existence_of_edge_length_limited_paths.py` | offline: sort queries by limit, union the edges lighter than it, answer in the original order |
| Course Schedule | `graphs/course_schedule.py` · `practice/simple/43_course_schedule.py` | Kahn: possible iff all n courses get popped |
| Course Schedule II | `graphs/course_schedule_ii.py` | Kahn's pop order is the answer; shorter than n means a cycle → [] |
| Critical Connections in a Network | `graphs/critical_connections_in_a_network.py` | Tarjan: tree edge (u, v) is a bridge iff low[v] > disc[u] |
| Find All People With Secret | `graphs/find_all_people_with_secret.py` | union one timestamp at a time; reset everyone not connected to person 0 |
| Find the Town Judge | `graphs/find_the_town_judge.py` | score = in-degree − out-degree; only the judge reaches n − 1 |
| Graph Valid Tree | `graphs/graph_valid_tree.py` | exactly n − 1 edges and no union fails |
| Largest Component Size by Common Factor | `graphs/largest_component_size_by_common_factor.py` | union each number with the first owner of each prime factor; count numbers per root |
| Minimize Malware Spread | `graphs/minimize_malware_spread.py` | removing a node saves its component only if it is the only infected one; pick the biggest (ties: smallest index; none: min(initial)) |
| Number of Connected Components in an Undirected Graph | `graphs/number_of_connected_components.py` | start at n; every successful union subtracts one |
| Number of Islands II | `graphs/number_of_islands_ii.py` | new land: +1, then −1 for each distinct neighbouring root it merges |
| Parallel Courses III | `graphs/parallel_courses_iii.py` | longest path in a DAG: push finish times forward in Kahn order |
| Reconstruct Itinerary | `graphs/reconstruct_itinerary.py` | Hierholzer: fly the smallest ticket, write an airport when stuck, reverse |
| Redundant Connection | `graphs/redundant_connection.py` · `practice/simple/47_redundant_connection.py` | the first union whose two ends already share a root |
| Redundant Connection II | `graphs/redundant_connection_ii.py` | a two-parent node? try dropping the later edge, else the earlier; otherwise the cycle edge |
| Remove Max Number of Edges to Keep Graph Fully Traversable | `graphs/remove_max_number_of_edges_to_keep_graph_fully_traversable.py` | two DSUs; shared edges first; removable = edges − kept |
| Sort Items by Groups Respecting Dependencies | `graphs/sort_items_by_groups_respecting_dependencies.py` | topo-sort items and groups separately; loners get their own group |

### Self-check

1. You reversed every edge in Course Schedule by mistake. Why does part I still pass while part II fails?
<details><summary>Answer</summary>Reversing every edge turns each cycle into a cycle and each acyclic graph into an acyclic graph, so "is there an order?" has the same answer. But the order you return is valid for the <em>reversed</em> graph: every course now comes out before its own prerequisites, so part II's answer is wrong.</details>

2. Kahn's algorithm stopped with `len(order) < n`. What can you say about the nodes left over?
<details><summary>Answer</summary>Each of them still has indegree at least 1, and every unfinished predecessor of a leftover node is itself a leftover. Walking backwards along those edges never has to stop, so in a finite graph it must repeat a node: there is a cycle among the leftovers. (Some leftovers may only sit downstream of the cycle.)</details>

3. What does `union(a, b)` returning False tell you, and which problems use exactly that?
<details><summary>Answer</summary>a and b were already connected before this edge, so the edge closes a cycle. Redundant Connection returns that edge, Graph Valid Tree fails on it, a component count ignores it, and Kruskal's MST (<a href="#s19">Graphs III</a>) skips it.</details>

4. When does union-find beat a BFS for "are a and b connected?"
<details><summary>Answer</summary>For a fixed graph and a single question, one BFS or DFS is simplest and just as fast. When edges arrive over time, when there are many questions between the arrivals, or when you need "the edge that closes a cycle", union-find wins: each operation is nearly O(1) and nothing is recomputed.</details>

<a id="s19"></a>

## Graphs III: Weighted Paths & MST

> Once edges have prices, "fewest edges" is no longer "cheapest". **Dijkstra** is BFS with a priority queue: always settle the closest unsettled node, because with no negative edges nobody can reach it more cheaply later. **A minimum spanning tree** connects everything for the least total price: keep taking the cheapest edge that does not close a loop.

[Graphs I](#s17) counted every move as one step, so the first arrival at a node was the cheapest. Once edges carry prices that stops being true: a heap takes over from the queue, and the union-find of [Graphs II](#s18) comes back to build spanning trees.

**Reach for it when** edges have weights and the problem asks for the minimum cost, time or effort to get somewhere: that is Dijkstra. When every weight is 0 or 1, a free move or a paid move, a deque replaces the heap, the 0-1 BFS. When a route is only as hard as its *worst* step, Dijkstra runs with `max` in place of `+`.

"Connect all points or cities as cheaply as possible" is a minimum spanning tree, built by Prim or Kruskal, and "at most k stops" or negative weights call for Bellman-Ford rounds.

**In this repo:** `graphs/` (7 of its 45 problems) · bank: `practice/simple/45_network_delay_time.py`, `practice/simple/46_min_cost_to_connect_all_points.py` · basics: `practice/simple/basics/graphs/06_dijkstra.py`, `practice/simple/basics/graphs/05_prim_mst.py`, `practice/simple/basics/graphs/04_kruskal_mst.py`, `practice/simple/basics/graphs/03_union_find.py`

### The picture

```text
         1
   A ---------- B          pop (0,A)  settle A  push B=1 C=5   heap (1,B) (5,C)
   |            |          pop (1,B)  settle B  push D=3       heap (3,D) (5,C)
 5 |            | 2        pop (3,D)  settle D  push C=4       heap (4,C) (5,C)   C got cheaper
   |            |          pop (4,C)  settle C                 heap (5,C)
   C ---------- D          pop (5,C)  stale: 5 > dist[C] = 4, skip it
         1
```

The heap always hands out the closest node nobody has settled yet. C was first priced at 5, straight from A, then found again at 4 along A, B, D, C; the old entry is not removed but skipped when it surfaces. That skip is the **stale check** (lazy deletion, as in [Heaps](07_Heaps_Intervals_Greedy.ipynb#s13)).

Why is a popped node final?

```text
   settled (popped)                    in the heap (priced, not final yet)
( src ... x ) --w-->  y  [price(y) >= d]  ... -->  u

u is popped with d = the smallest price in the heap. Any other route to u leaves the
settled blob through some edge x -> y; at y it already costs >= price(y) >= d, and the
rest of the route adds >= 0 (no negative edges). Nothing can beat d.
```

```text
    0 ---4--- 1                 Kruskal: cheapest edge first; skip it if its ends are already connected
    |       / |
    1     2   5                 0-2 (1)  take   {0,2} {1} {3} {4}
    |   /     |                 1-2 (2)  take   {0,1,2} {3} {4}
    2 ---8--- 3 ---3--- 4       3-4 (3)  take   {0,1,2} {3,4}
                                0-1 (4)  SKIP   0 and 1 are already connected: it would close a loop
                                1-3 (5)  take   {0,1,2,3,4}: n - 1 = 4 edges, done, total 11

    Prim grows one blob from 0 instead, always adding the cheapest edge that leaves it:
    0-2 (1), 2-1 (2), 1-3 (5), 3-4 (3): the same total, 11
```

**Why the greedy edge is safe.** The argument is the cut property. Split the nodes into two sides and let e be the cheapest edge crossing the split. Take any best tree that does not use e: adding e closes a loop, and that loop must cross the split a second time through some edge f with w(f) ≥ w(e). Swap f for e: the tree still spans everything and is never heavier, so some best tree uses e.

Prim uses the split "tree so far | the rest", so the cheapest edge leaving its tree is always safe to add.

Kruskal, at an edge (u, v) whose ends have different roots, uses the split "u's group | the rest". Every cheaper edge has already been processed, and none of them can still leave u's group, because it would have been taken; so (u, v) is the cheapest edge leaving that group, and safe.

**Why it is fast.** To relax an edge u → v is to ask whether going through u makes v cheaper, and to lower v's price if it does. Bellman-Ford relaxes every edge in V − 1 rounds, O(V · E), because it does not know which distances are final. Dijkstra settles nodes in order of distance, so each node's edges are relaxed once, and the heap hands out the next node in O(log V): O((V + E) log V).

For the MST, the brute force tries spanning trees, and there are exponentially many. The cut property makes the greedy choice safe, so one sorted pass (Kruskal) or one heap-driven growth (Prim) is enough.

### From idea to code

**The idea in one sentence:** *pop the cheapest entry from a min-heap; if it is out of date, skip it; otherwise it is final, so offer its neighbours at their new prices.* Dijkstra prices a node by the whole path `d + w`; Prim prices it by the single edge `w` that would attach it to the tree. That is the one real difference between the two loops: both pop the cheapest entry, skip the out-of-date ones, and offer the neighbours.

For Dijkstra the **state** is `dist` and a min-heap of `(dist, node)` entries. By **definition**, `dist[v]` is the cheapest path from `src` to v found so far, and it is final once v is popped with `d == dist[v]`. The **invariant**: nodes are settled in nondecreasing distance, and a settled distance never changes. A **step** pops `(d, u)`, skips it if `d > dist[u]`, and relaxes each edge `(v, w)`: if `d + w < dist[v]`, it lowers `dist[v]` and pushes the new price.

The **record** is `dist[u]`, final at its non-stale pop. **Init** is `dist = {src: 0}` and the heap `[(0, src)]`. The **return** is `dist`, `dist[target]` or `max(dist.values())`, and −1 when a node never appears.

Prim makes the same decisions about a tree. Its state is `in_tree` and a min-heap of `(edge cost, node)`, where an entry `(w, v)` means "v could join the tree through an edge of cost w". Its invariant is that the tree built so far is part of some minimum spanning tree. A step pops `(w, v)`, skips it if v is already in the tree, adds v, and pushes `(w', x)` for every edge to an outside node x.

Prim records `total += w` when v joins. Init is the heap `[(0, start)]`, so the start joins for free, and the return is `total` once all n have joined; fewer means the graph is disconnected.

The sentences map onto lines as usual. "The closest node not settled yet" is `d, u = heapq.heappop(heap)`, and "an out-of-date copy of u" is `if d > dist[u]: continue`. "Going through u is cheaper" is `if d + w < dist.get(v, math.inf):`, and "remember it and offer it" is `dist[v] = d + w` followed by a push.

Network Delay Time sends a signal from node k along directed edges, each with a travel time, and asks when the last node hears it, or -1 if some node never does: with the edges 2 → 1, 2 → 3 and 3 → 4, each taking 1, a signal from 2 needs 2. So `dijkstra` computes every shortest distance and the wrapper takes the largest. The whole point is the order of two moments: a node is final when it is *popped* past the stale check, never when it is pushed.

In [ ]:
def dijkstra(graph, src):
    """graph[u] = [(v, w), ...] with every w >= 0, and every node is a key (use defaultdict(list)).
    Returns {node: shortest distance from src} for every node src can reach."""
    dist = {src: 0}                               # STATE + INIT: dist[v] = cheapest way to v found so far
    heap = [(0, src)]                             # STATE + INIT: (price, node) entries, cheapest on top
    while heap:
        d, u = heapq.heappop(heap)                # the cheapest entry left
        if d > dist[u]:
            continue                              # stale: u was already settled closer
        # RECORD: d == dist[u] is final now; a single target is checked here: if u == target: return d
        for v, w in graph[u]:
            if d + w < dist.get(v, math.inf):     # relax: going through u is cheaper
                dist[v] = d + w                   # STEP: v's best price so far...
                heapq.heappush(heap, (d + w, v))  # ...offered to the heap
    return dist                                   # RETURN: unreachable nodes are simply missing


def network_delay_time(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = dijkstra(graph, k)
    return max(dist.values()) if len(dist) == n else -1     # the last node to hear it, or -1


print(network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2))             # 2
print(network_delay_time([[1, 2, 4], [1, 3, 1], [3, 2, 1], [2, 4, 2]], 4, 1))  # 4  (1 -> 3 -> 2 beats 1 -> 2)
print(network_delay_time([[1, 2, 1]], 2, 2))                                   # -1 (nothing leaves node 2)

**Try it**
- Add `print(d, u)` right after the `heappop` and run the second example: node 2 comes out twice, at 2 (real) and at 4 (stale, skipped).
- The BFS habit: keep a `seen` set, add `v` when you push it, and skip neighbours already seen. The second example answers 6 instead of 4: node 2 was locked in at 4 by the direct edge before 1 → 3 → 2 (cost 2) was found. A Dijkstra node is final when it is popped.
- Change `d > dist[u]` to `d >= dist[u]`: all three examples answer -1. Even the very first entry, `(0, src)`, now looks stale.
- Give `dijkstra` a `target` and return at discovery (`if v == target: return d + w` right after `dist[v] = d + w`): from 1 to 2 in the second example you get 4 instead of 2. Discovery-time checks are safe in BFS, where every edge costs 1, not here.

Min Cost to Connect All Points asks for the cheapest way to connect points on a plane, where joining two points costs their Manhattan distance, `|x1 − x2| + |y1 − y2|`: the five points below cost 20. It is a minimum spanning tree of the complete graph, so there is no edge list; the "edges" from a point are its distances to every point still outside the tree. "The cheapest edge leaving the tree" is a pop that skips any point already in.

In [ ]:
def min_cost_connect_points(points):
    n = len(points)
    in_tree = [False] * n                         # STATE + INIT: who has joined the tree
    heap = [(0, 0)]                               # STATE + INIT: (cost to join, point); point 0 joins free
    total = joined = 0
    while joined < n:                             # all points connect to all, so the heap never runs dry
        cost, i = heapq.heappop(heap)             # the cheapest edge leaving the tree
        if in_tree[i]:
            continue                              # stale: i joined earlier through a cheaper edge
        in_tree[i] = True                         # STEP: i joins the tree...
        total += cost                             # RECORD: ...through the cheapest crossing edge, a safe one
        joined += 1
        for j in range(n):                        # offer i's edges to every point still outside
            if not in_tree[j]:
                manhattan = abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1])
                heapq.heappush(heap, (manhattan, j))
    return total                                  # RETURN


print(min_cost_connect_points([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]))   # 20
print(min_cost_connect_points([[3, 12], [-2, 5], [-4, 1]]))                # 18

**Try it**
- Delete the `if in_tree[i]: continue` lines: the first example answers 18. Point 3 is popped a second time and counted as a fifth join, so the loop stops before point 2 ever joins.
- Start the heap with `(5, 0)` instead of `(0, 0)`: 25. The start must join for free.
- Print `i, cost` when a point joins: 0 (0), 1 (4), 3 (3), 4 (4), 2 (9). The costs are not sorted: Prim takes the cheapest edge leaving the *current* tree, not the cheapest edge overall.

Kruskal builds the same kind of tree from an edge list: sort all edges once, cheapest first, and let union-find, from [Graphs II](#s18), say whether an edge would close a loop, `find(u) == find(v)`. On the picture's graph it keeps four edges of total weight 11, and it stops as soon as it has n − 1.

In [ ]:
def kruskal(n, edges):
    """edges = [(u, v, w)]. Returns (total weight, edges kept)."""
    parent = list(range(n))                       # STATE + INIT: everyone is their own group

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]         # path halving
            x = parent[x]
        return x

    total, kept = 0, []                           # STATE + INIT: the forest built so far
    edges = sorted(edges, key=lambda e: e[2])     # INIT: cheapest edge first
    for u, v, w in edges:
        ru, rv = find(u), find(v)
        if ru == rv:
            continue                              # both ends already connected: it would close a loop
        parent[ru] = rv                           # STEP: the edge merges two groups
        total += w                                # RECORD
        kept.append((u, v, w))
        if len(kept) == n - 1:
            break                                 # a spanning tree has exactly n - 1 edges
    return total, kept                            # RETURN: fewer than n - 1 edges = disconnected


print(kruskal(5, [(0, 1, 4), (0, 2, 1), (1, 2, 2), (1, 3, 5), (2, 3, 8), (3, 4, 3)]))
# (11, [(0, 2, 1), (1, 2, 2), (3, 4, 3), (1, 3, 5)])

**Try it**
- Delete the `if ru == rv: continue` lines: you get `(10, [(0, 2, 1), (1, 2, 2), (3, 4, 3), (0, 1, 4)])`, four edges that contain the loop 0-1-2 and leave 3 and 4 cut off from the rest.
- Sort by `-e[2]` (heaviest first): a *maximum* spanning tree, weight 20.
- `kruskal(4, [(0, 1, 1), (2, 3, 1)])` keeps only 2 edges: fewer than `n - 1` means the graph is disconnected.

### Watch it work

The trace runs Dijkstra on the picture's graph and prints one line per pop: a pop either settles its node and offers the neighbours, or is skipped as stale. Watch C come out twice, once at its old price and once at its real one.

In [ ]:
def trace_dijkstra(graph, src):
    dist, heap = {src: 0}, [(0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist[u]:
            print(f"pop ({d}, {u})  stale: dist[{u}] is already {dist[u]}, skip")
            continue
        pushed = []
        for v, w in graph[u]:
            if d + w < dist.get(v, math.inf):
                dist[v] = d + w
                heapq.heappush(heap, (d + w, v))
                pushed.append(f"{v}={d + w}")
        print(f"pop ({d}, {u})  settle {u}  push {pushed}  heap={sorted(heap)}")


roads = {"A": [("B", 1), ("C", 5)], "B": [("A", 1), ("D", 2)],
         "C": [("A", 5), ("D", 1)], "D": [("B", 2), ("C", 1)]}
trace_dijkstra(roads, "A")

**Try it**
- Make the road A-C cost 3 instead of 5 (in both directions): no stale entry this time, because C's first price is already its best.
- Add `last = max(last, d)` right after the pop, *before* the stale check (start with `last = 0`), and print it at the end: 5, although the farthest node is only 4 away. Anything recorded per pop belongs after the stale check.
- Run `trace_dijkstra({"A": [("B", 1), ("C", 2)], "B": [("D", 1)], "C": [("B", -2)], "D": []}, "A")`: B is settled twice, first at 1 and later at 0. A negative edge broke "popped means final".

### Where it goes wrong

1. **Marking a node when it is pushed (the BFS habit).** A cheaper route can still arrive through a node popped later. With `seen` on push, the second delay example answers 6 instead of 4 (plain BFS gives the same 6).
2. **Stopping when the target is discovered.** Its first price may not be its best: from 1 to 2 in the second delay example you get 4 instead of 2. Return when the target is *popped*.
3. **No stale check while recording at pop time.** Distances still come out right (an out-of-date `d` cannot improve anything), but whatever you record per pop is wrong: the max over pops in the trace graph is 5 instead of 4, and Prim counts a point twice (18 instead of 20).
4. **Negative edges.** "Popped means final" needs `w >= 0`. With `A->B 1, A->C 2, C->B -2, B->D 1`, B is popped at 1 before the cheaper route through C (0) is found. A version that never re-expands a node (a `done` set) leaves D at 2 instead of 1; this lazy version re-pushes B and recovers here, but can take exponential time in general. Negative weights need Bellman-Ford.
5. **0-1 BFS with a plain FIFO.** A free move must go to the *front*; appending it lets a cost-1 entry come out first, and `[[1, 1, 3], [3, 2, 2], [1, 1, 4]]` answers 1 instead of 0.
6. **Bottleneck with the wrong key.** Pushing the step alone instead of `max(t, step)` forgets the worst step so far: Swim in Rising Water, the lowest water level at which you can swim from one corner to the other, answers 6 instead of 16 on the 5 × 5 grid in Variations.
7. **Unreachable nodes.** They never get a `dist`. Check `len(dist) == n` before taking a max, and return −1 otherwise: without the check, `network_delay_time([[1, 2, 1]], 3, 1)` answers 1, although node 3 never hears the signal.
8. **Nodes missing from the graph dict.** `dijkstra({0: [(1, 1)]}, 0)` raises `KeyError: 1`, because node 1 has no entry. Build the graph as a `defaultdict(list)`.
9. **Prim's start with a cost.** Pushing `(5, 0)` instead of `(0, 0)` adds 5 to the total (25 instead of 20). The start joins for free.
10. **Prim on a disconnected edge list.** `while joined < n` pops from an empty heap once the reachable part is used up: three nodes with the single edge 0-1 raise `IndexError` after two joins. Loop `while heap and joined < n`, and return −1 if `joined < n`.
11. **Losing edge indexes.** Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree must name edges by their input index. Sorting `[[0, 1, 2], [1, 2, 1]]` in place moves the weight-1 edge to index 0, so it is reported as edge 0 when the input called it edge 1. Sort a list of indexes.
12. **Unorderable heap entries.** On a tie in distance, Python compares the next item of the tuple: pushing `(1, obj)` twice with plain objects raises `TypeError`. Push `(dist, index)` or `(dist, counter, obj)`, as in [Heaps](07_Heaps_Intervals_Greedy.ipynb#s13).

### Edge cases to say out loud

The source alone · an unreachable node (−1) · zero-weight edges · parallel edges (the cheaper one wins) · self-loops (harmless) · one point (MST cost 0) · duplicate points (cost 0) · a disconnected graph for an MST (fewer than n − 1 edges) · equal weights (several MSTs, one total).

In [ ]:
assert dijkstra({0: []}, 0) == {0: 0}                                  # the source alone
assert dijkstra({0: [(1, 0)], 1: []}, 0) == {0: 0, 1: 0}               # zero-weight edges are fine
assert dijkstra({0: [(1, 5), (1, 2)], 1: []}, 0)[1] == 2               # parallel edges: the cheaper one wins
assert network_delay_time([], 1, 1) == 0                                # one node hears it at time 0
assert network_delay_time([[1, 2, 1]], 3, 1) == -1                      # node 3 never hears it
assert min_cost_connect_points([[1, 1]]) == 0                           # one point: nothing to connect
assert min_cost_connect_points([[0, 0], [0, 0]]) == 0                   # duplicate points cost 0
assert kruskal(3, [(0, 1, 1)]) == (1, [(0, 1, 1)])                      # disconnected: fewer than n - 1 edges
print("edge cases pass")

**Try it**
- Predict `dijkstra({0: [(1, 3)], 1: [(0, 3)]}, 1)` before running it: `{1: 0, 0: 3}`.
- `dijkstra({0: [(0, 5)]}, 0)` is still `{0: 0}`: a loop can never make a path cheaper.
- `min_cost_connect_points([[0, 0], [1, 0], [2, 0]])` is 2: three points on a line, and the tree is the line.

### Variations

Every variation keeps the pop, skip and relax loop and changes one thing: the container, the key, the number of rounds, or what a node is.

| Variation | What changes from the template | Problems |
|---|---|---|
| **All distances, then a max** | run to the end; answer = `max(dist.values())`, −1 if a node is missing | Network Delay Time (743) |
| **Weights are only 0 or 1** | a deque instead of a heap: weight 0 → `appendleft`, weight 1 → `append` | Minimum Cost to Make at Least One Valid Path in a Grid (1368), Minimum Moves to Move a Box to Their Target Location (1263) |
| **Dijkstra on a grid or on states** | a node is `(r, c)` or a whole state like `(box, player)`; the loop is unchanged | Path With Minimum Effort (1631), Swim in Rising Water (778), Minimum Moves to Move a Box to Their Target Location (1263) |
| **Minimise the worst step** | key = `max(so_far, step)` instead of a sum | Path With Minimum Effort (1631), Swim in Rising Water (778) |
| **Most likely path** | push `-p`: a product of probabilities ≤ 1 never grows along a path | Path with Maximum Probability (1514) |
| **At most k edges, or negative weights** | Bellman-Ford: k + 1 rounds, each relaxing every edge from a copy of last round's prices | Cheapest Flights Within K Stops (787) |
| **MST on a set of points** | Prim with a heap, or the O(n²) array version on a complete graph | Min Cost to Connect All Points (1584) |
| **MST from an edge list** | Kruskal: sort + union-find | Min Cost to Connect All Points (1584), with every pair listed as an edge |
| *Second pass:* **several fixed endpoints** | Dijkstra from each source, and from the target on reversed edges; minimise over the meeting node | Minimum Weighted Subgraph With the Required Paths (2203) |
| *Second pass:* **which edges matter to the MST** | rerun Kruskal once per edge, skipping it or forcing it in first | Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree (1489) |

Two weight patterns come first, and the simpler one lets a deque replace the heap. When every weight is 0 or 1, the heap only ever holds two distances, `d` and `d + 1`, and a deque keeps that order for free: a free move joins the *front* of the line with `dq.appendleft(...)`, a paid move the back with `dq.append(...)`.

Minimum Cost to Make at Least One Valid Path in a Grid puts an arrow in every cell: right, left, down or up. Following a cell's arrow is free, turning it to point elsewhere costs 1, and the problem asks for the cheapest way to make a path from the top-left to the bottom-right corner: 3 for the first grid below.

In [ ]:
ARROWS = {1: (0, 1), 2: (0, -1), 3: (1, 0), 4: (-1, 0)}   # sign in the cell -> direction it points


def min_cost_valid_path(grid):
    rows, cols = len(grid), len(grid[0])
    dist = {(0, 0): 0}
    dq = deque([(0, 0, 0)])                         # (cost, r, c): only two cost values live in here, d and d + 1
    while dq:
        d, r, c = dq.popleft()
        if d > dist[(r, c)]:
            continue                                # stale, exactly like Dijkstra
        if (r, c) == (rows - 1, cols - 1):
            return d
        for sign, (dr, dc) in ARROWS.items():
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                w = 0 if grid[r][c] == sign else 1  # follow the arrow for free, turn it for 1
                if d + w < dist.get((nr, nc), math.inf):
                    dist[(nr, nc)] = d + w
                    if w == 0:
                        dq.appendleft((d, nr, nc))  # same cost: jump the line
                    else:
                        dq.append((d + 1, nr, nc))  # one more: back of the line


print(min_cost_valid_path([[1, 1, 1, 1], [2, 2, 2, 2], [1, 1, 1, 1], [2, 2, 2, 2]]))   # 3
print(min_cost_valid_path([[1, 1, 3], [3, 2, 2], [1, 1, 4]]))                           # 0
print(min_cost_valid_path([[1, 2], [4, 3]]))                                            # 1

**Try it**
- Change `dq.appendleft(...)` to `dq.append(...)`, a plain FIFO: `[[1, 1, 3], [3, 2, 2], [1, 1, 4]]` now answers 1 instead of 0. A cost-1 entry got out before the free path arrived.
- Print `[x[0] for x in dq]` just before each `popleft()` on the first grid: only two values ever, `d` and `d + 1`, in order. That is why a deque can replace the heap.
- Swap the deque for `heapq` (push `(d + w, nr, nc)`, pop the smallest): plain Dijkstra, the same three answers, with an extra log factor.

The same deque works on states. Minimum Moves to Move a Box to Their Target Location asks for the fewest pushes that bring a box onto its target, while the player walks for free. A node is `(box, player)`: a player step that does not touch the box costs 0, and walking into the box pushes it one cell for a cost of 1, if the cell beyond is free. The answer is the cost of the first popped state whose box sits on the target.

The second pattern changes the key instead of the container. Path With Minimum Effort asks for the route between the top-left and bottom-right cells whose steepest step, the largest height difference between neighbours, is as small as possible: 2 for the first grid below. Swim in Rising Water asks for the lowest water level at which you can swim between the same corners, a cell being open once the water reaches its height: 16 for the 5 × 5 grid.

Dijkstra still works, because `max` never decreases along a path, just as `+` never does with non-negative weights. "A path is as bad as its worst step" is `max(t, step)` in place of `d + w`, and only that key changes, so one helper serves both problems.

In [ ]:
def bottleneck_path(rows, cols, start_key, step_key):
    """Smallest possible worst step on a path from (0, 0) to (rows - 1, cols - 1)."""
    best = {(0, 0): start_key}                    # best[cell] = lowest "worst step so far" on a path to it
    heap = [(start_key, 0, 0)]
    while heap:
        t, r, c = heapq.heappop(heap)
        if t > best[(r, c)]:
            continue                              # stale
        if (r, c) == (rows - 1, cols - 1):
            return t                              # popped = final, as in Dijkstra
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                nt = max(t, step_key((r, c), (nr, nc)))   # a path is as hard as its worst step: max, not +
                if nt < best.get((nr, nc), math.inf):
                    best[(nr, nc)] = nt
                    heapq.heappush(heap, (nt, nr, nc))


def minimum_effort(h):                            # 1631: a step costs the height difference
    return bottleneck_path(len(h), len(h[0]), 0, lambda a, b: abs(h[b[0]][b[1]] - h[a[0]][a[1]]))


def swim_in_water(g):                             # 778: a step costs the height of the cell you enter
    return bottleneck_path(len(g), len(g), g[0][0], lambda a, b: g[b[0]][b[1]])


print(minimum_effort([[1, 2, 2], [3, 8, 2], [5, 3, 5]]), minimum_effort([[1, 2, 3], [3, 8, 4], [5, 3, 5]]))   # 2 1
print(swim_in_water([[0, 2], [1, 3]]), swim_in_water([[0, 1, 2, 3, 4], [24, 23, 22, 21, 5], [12, 13, 14, 15, 16],
                                                      [11, 17, 18, 19, 20], [10, 9, 8, 7, 6]]))                  # 3 16

**Try it**
- Push the step alone (`nt = step_key(...)`, no `max`): the 5×5 Swim in Rising Water grid answers 6 instead of 16. The key forgot the worst step on the way.
- Use `t + step_key(...)` instead: the 5×5 grid gives 57, the cheapest *total* climb, which is a different problem.
- `swim_in_water([[3, 2], [0, 1]])` is 3: the start's own height counts, so you cannot leave before the water reaches it. `minimum_effort([[5]])` is 0: one cell, no step at all.

Path With Minimum Effort has two other accepted answers. One is binary search on the answer t, with one BFS or DFS per guess asking "can I reach the corner using steps ≤ t?" ([Binary Search](05_Binary_Search_Sorting.ipynb#s09)). The other is a Kruskal-style sweep that adds edges by weight until the start and the corner share a root.

Path With Maximum Probability (1514) asks for the most likely path between two nodes when each edge succeeds with its own probability, and it is Dijkstra too. A product of probabilities ≤ 1 never grows along a path, so push `-p` and pop the most likely node first.

A cap on the number of edges breaks plain Dijkstra, and rounds handle it. Cheapest Flights Within K Stops asks for the cheapest price from `src` to `dst` with at most k stops, or -1: on the first network below, k = 1 allows 0 → 1 → 3 for 700. Dijkstra keeps one price per node, so it cannot also cap the number of edges unless the node becomes (node, edges used).

Bellman-Ford counts edges for free. Each round relaxes *every* edge once, reading last round's prices, so after round i every node knows its cheapest route with at most i edges. It also tolerates negative weights: V − 1 rounds find plain shortest paths in O(V · E).

In [ ]:
def find_cheapest_price(n, flights, src, dst, k):
    dist = [math.inf] * n
    dist[src] = 0
    for _ in range(k + 1):                 # at most k stops = at most k + 1 flights
        new = dist[:]                      # read last round's prices, write this round's
        for u, v, w in flights:
            if dist[u] + w < new[v]:
                new[v] = dist[u] + w
        dist = new
    return -1 if dist[dst] == math.inf else dist[dst]


print(find_cheapest_price(4, [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], 0, 3, 1))   # 700
print(find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1),
      find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0))                             # 200 500

**Try it**
- Relax in place (`new = dist` instead of `dist[:]`): with k = 0 the second network answers 200 instead of 500. The flights 0 → 1 and 1 → 2 both slipped into one round, which means two flights for a "zero stops" ticket.
- Allow k = 2 on the first network: 400, along 0 → 1 → 2 → 3.
- `find_cheapest_price(3, [[0, 1, 100]], 0, 2, 1)` is -1: nothing ever flies into 2.

On a complete graph the heap can go. There are n² edges anyway, so a heap only adds a log factor: keep `best[j]`, the cheapest edge from the tree to point j, and pick the next point with a plain scan. The five points of Min Cost to Connect All Points cost 20 again, and the three points `[[3, 12], [-2, 5], [-4, 1]]` cost 18.

In [ ]:
def min_cost_connect_points_dense(points):
    n = len(points)
    best = [math.inf] * n                         # best[j] = cheapest edge from the tree to j so far
    best[0] = 0
    in_tree = [False] * n
    total = 0
    for _ in range(n):
        i = min((j for j in range(n) if not in_tree[j]), key=lambda j: best[j])   # an O(n) scan replaces the heap
        in_tree[i] = True
        total += best[i]
        for j in range(n):
            if not in_tree[j]:
                best[j] = min(best[j], abs(points[i][0] - points[j][0]) + abs(points[i][1] - points[j][1]))
    return total


print(min_cost_connect_points_dense([[0, 0], [2, 2], [3, 10], [5, 2], [7, 0]]),
      min_cost_connect_points_dense([[3, 12], [-2, 5], [-4, 1]]))                # 20 18

**Try it**
- Forget `best[0] = 0`: the answer becomes `inf`. The first scan still picks point 0, but it joins at an infinite price.
- Print `best` each time a point joins on the 3-point example: `[0, inf, inf]`, `[0, 12, 18]`, `[0, 12, 6]`. An entry only ever goes down: point 2's price drops from 18 to 6 once point 1 is in.
- Compare both versions on random point sets: they always agree. The array version does n² simple steps and keeps no stale entries.

The rest of this section is a second pass: Hard problems that reuse the same moves. Skip them until the main path is automatic.

Minimum Weighted Subgraph With the Required Paths gives a weighted directed graph and asks for the lightest set of edges through which both `src1` and `src2` can reach `dest`, or -1: 9 for the graph below. The two paths meet at some node x and share the rest, so the cost is `d1[x] + d2[x] + (x → dest)`. Distances from the sources are two ordinary Dijkstras, and "from every x to dest" sounds like n searches, but on the reversed graph it is a single Dijkstra *from* `dest`.

In [ ]:
def minimum_weight(n, edges, src1, src2, dest):
    forward, backward = defaultdict(list), defaultdict(list)
    for u, v, w in edges:
        forward[u].append((v, w))
        backward[v].append((u, w))                  # reversed: Dijkstra from dest measures every "x -> dest"
    d1, d2, to_dest = dijkstra(forward, src1), dijkstra(forward, src2), dijkstra(backward, dest)
    best = min((d1[x] + d2[x] + to_dest[x] for x in range(n) if x in d1 and x in d2 and x in to_dest),
               default=math.inf)                    # the two paths meet at x, then share x -> dest
    return -1 if best == math.inf else best


print(minimum_weight(6, [[0, 2, 2], [0, 5, 6], [1, 0, 3], [1, 4, 5], [2, 1, 1], [2, 3, 3], [2, 3, 4],
                         [3, 4, 2], [4, 5, 1]], 0, 1, 5))   # 9
print(minimum_weight(3, [[0, 1, 1], [2, 1, 1]], 0, 1, 2))  # -1

**Try it**
- Run the third Dijkstra on `forward` instead of `backward`: 12. Nothing can be reached *from* node 5, so the only meeting point left is 5 itself.
- Print `d1[x]`, `d2[x]` and `to_dest[x]` for every x: meeting at 0 or at 1 both cost 9 (`0 + 3 + 6` and `3 + 0 + 6`).
- `minimum_weight(3, [[0, 2, 5], [1, 2, 5], [0, 1, 1]], 0, 1, 2)` is 6: src1 rides to src2 for 1, then they share 1 → 2. The meeting point can be one of the sources.

Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree asks which edges every minimum spanning tree uses, the critical ones, and which only some of them use, the pseudo-critical ones. Kruskal is cheap, so run it once per question about one edge.

An edge is critical when skipping it makes the tree heavier, or impossible, where a run that cannot span counts as infinitely heavy. An edge that is not critical is pseudo-critical when forcing it in first still gives the best weight. Sort a list of indexes, not the edges themselves, so that the answer keeps the input's numbering (trap 11).

### Say it in the interview

> "The edges have different costs, so 'fewest edges' isn't 'cheapest' and plain BFS is out. Bellman-Ford would relax every edge V − 1 times, O(V · E). The weights are non-negative, so I'll use Dijkstra: a min-heap of (distance, node)."

> "When I pop a node with distance d, any other route to it must leave the settled nodes through something still in the heap, which already costs at least d, and edges can't subtract, so d is final and I relax its edges once. Entries that went out of date are skipped when they surface. With lazy deletion the heap holds up to E entries: O(E log E) = O(E log V)."

> "We need every point connected at minimum total cost, which is a minimum spanning tree. The cheapest edge leaving the tree built so far is always safe, so Prim's grows the tree with a heap of (edge cost, point): O(n² log n) on the complete graph of n points, or O(n²) with the array version."

Point at the stale check, the relax condition, and the line where the answer is read. If the weights are only 0 and 1, say "0-1 BFS" and show the two `append` calls. Follow-ups to expect: negative weights (Bellman-Ford), at most k stops (rounds with a copy), the path itself (`parent[v] = u` on every successful relax, then walk back), a single target (return at its first non-stale pop), and a complete graph (the O(V²) array version).

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree | `graphs/find_critical_and_pseudo_critical_edges_in_minimum_spanning_tree.py` | critical: the MST without it is heavier; pseudo: forcing it in first keeps the MST weight |
| Min Cost to Connect All Points | `graphs/min_cost_to_connect_all_points.py` · `practice/simple/46_min_cost_to_connect_all_points.py` | Prim on the complete graph: heap of (distance, point), skip points already in; or O(n²) with an array |
| Minimum Cost to Make at Least One Valid Path in a Grid | `graphs/minimum_cost_to_make_at_least_one_valid_path_in_a_grid.py` | 0-1 BFS: following the arrow costs 0 (front), turning it costs 1 (back) |
| Minimum Moves to Move a Box to Their Target Location | `graphs/minimum_moves_to_move_a_box_to_their_target_location.py` | 0-1 BFS over (box, player) states: walking costs 0, a push costs 1 |
| Minimum Weighted Subgraph With the Required Paths | `graphs/minimum_weighted_subgraph_with_the_required_paths.py` | Dijkstra from src1, src2, and from dest on reversed edges; minimise over the meeting node |
| Network Delay Time | `graphs/network_delay_time.py` · `practice/simple/45_network_delay_time.py` | Dijkstra from k; answer = the largest distance, −1 if a node is never reached |
| Swim in Rising Water | `graphs/swim_in_rising_water.py` | bottleneck Dijkstra: key = the highest elevation so far; the first pop of the corner wins |

### Self-check

1. Why is a node's distance final when it is popped (and not stale)? What breaks with a negative edge?
<details><summary>Answer</summary>Every other route to that node has to leave the settled region through some node still in the heap, and that node is already at least as far away. Non-negative edges can only add to that, so no other route can be cheaper. A negative edge can subtract, so a route through a node popped <em>later</em> can still undercut a settled distance.</details>

2. In 0-1 BFS, why must a free move go to the front of the deque?
<details><summary>Answer</summary>The deque must stay sorted by cost, like a heap. A free move has the same cost <code>d</code> as the node being expanded, which is the smallest cost in the deque, so it belongs in front of every <code>d + 1</code> entry. Put at the back, it waits behind more expensive entries, and the target can be popped with a cost that is too high.</details>

3. Prim and Dijkstra look almost identical. What is the one real difference?
<details><summary>Answer</summary>The key in the heap. Dijkstra orders nodes by the whole path cost <code>d + w</code> from the source; Prim orders them by the single edge <code>w</code> that would attach them to the tree.</details>

4. Why can't Dijkstra mark a node when it is pushed, as BFS does?
<details><summary>Answer</summary>In BFS every edge costs 1, so the first discovery comes from the closest ring and is already final. With weights, a node first discovered through an expensive edge can still be reached more cheaply through a node that is popped later. Marking it at push time blocks that cheaper route: the second delay example then answers 6 instead of 4.</details>